# Phase 3 — CIC-IDS2018 Feature Engineering

## Project: NetVision-AI-SIH26153

This notebook performs feature engineering on the cleaned and chronologically
ordered CIC-IDS2018 network traffic dataset produced in Phase 2.

### Input
Phase 2 processed dataset stored in the private Hugging Face repository:

`PrakulShetty/NetVision-AI-SIH26153`

Path:

`processed/phase2/`

### Output
Feature-engineered dataset:

`processed/phase3/`

### Phase 3 Objective
To prepare meaningful, non-redundant network traffic features for the
subsequent temporal representation and machine learning stages of the
NetVision-AI-SIH26153 project.

### Phase 3 Principle

Phase 2 data will remain unchanged. All Phase 3 transformations will produce
a separate Phase 3 dataset.

## 3.1 Environment and Library Setup

This section prepares the Python environment required for Phase 3 feature
engineering.

The main libraries used are:

- NumPy — numerical operations
- Pandas — data handling and analysis
- DuckDB — efficient processing of large CSV files
- Matplotlib — visualization
- Seaborn — statistical visualization
- Hugging Face Hub — access to the project dataset repository

No dataset transformation is performed in this step.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.1
# ENVIRONMENT AND LIBRARY SETUP
# ======================================================================

# Install required packages if they are not already available
!pip -q install duckdb huggingface_hub

# ----------------------------------------------------------------------
# Import libraries
# ----------------------------------------------------------------------

import os
import shutil
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    list_repo_files
)

# ----------------------------------------------------------------------
# Display environment information
# ----------------------------------------------------------------------

print("=" * 70)
print("PHASE 3 — ENVIRONMENT SETUP")
print("=" * 70)
print()

print("NumPy version       :", np.__version__)
print("Pandas version      :", pd.__version__)
print("DuckDB version      :", duckdb.__version__)
print("Environment         : Google Colab")
print()

print("Required libraries imported successfully. ✅")

print()
print("=" * 70)
print("✅ STEP 3.1 COMPLETE")
print("=" * 70)

PHASE 3 — ENVIRONMENT SETUP

NumPy version       : 2.1.3
Pandas version      : 2.2.3
DuckDB version      : 1.3.2
Environment         : Google Colab

Required libraries imported successfully. ✅

✅ STEP 3.1 COMPLETE


## 3.2 Hugging Face Authentication and Repository Connection

This section establishes the connection between the Phase 3 notebook and the
private Hugging Face dataset repository used by the NetVision-AI-SIH26153
project.

The repository contains:

- Original CIC-IDS2018 dataset files
- Phase 2 cleaned and chronologically ordered datasets
- Future Phase 3 feature-engineered datasets

For Phase 3, the primary input will be:

`processed/phase2/`

Authentication is performed using the existing Hugging Face login session in
Google Colab. No access token is stored directly in the notebook.

In [ ]:
# ======================================================================
# HUGGING FACE AUTHENTICATION
# ======================================================================

from huggingface_hub import login

login()

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.2
# HUGGING FACE AUTHENTICATION & REPOSITORY CONNECTION
# ======================================================================

# Project Hugging Face dataset repository
HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"

# Phase 2 dataset location — this will be our Phase 3 input
PHASE2_PATH = "processed/phase2"

# Phase 3 output location — created later in the workflow
PHASE3_PATH = "processed/phase3"

# ----------------------------------------------------------------------
# Verify existing Hugging Face authentication
# ----------------------------------------------------------------------

print("=" * 70)
print("PHASE 3 — HUGGING FACE CONNECTION")
print("=" * 70)
print()

print("1. Checking Hugging Face authentication...")

try:
    user_info = HfApi().whoami()
    username = user_info.get("name", "Unknown")

    print("   Authentication : SUCCESSFUL ✅")
    print("   Account        :", username)

except Exception as e:
    raise RuntimeError(
        "Hugging Face authentication failed. "
        "Please authenticate in Colab before continuing."
    ) from e

# ----------------------------------------------------------------------
# Verify project repository
# ----------------------------------------------------------------------

print()
print("2. Checking project repository...")

try:
    repo_files = list_repo_files(
        repo_id=HF_REPO,
        repo_type="dataset"
    )

    print("   Repository     :", HF_REPO)
    print("   Repository     : ACCESSIBLE ✅")
    print("   Total files    :", len(repo_files))

except Exception as e:
    raise RuntimeError(
        "Unable to access the project Hugging Face dataset repository."
    ) from e

# ----------------------------------------------------------------------
# Check Phase 2 directory
# ----------------------------------------------------------------------

print()
print("3. Checking Phase 2 dataset directory...")

phase2_files = [
    path for path in repo_files
    if path.startswith(PHASE2_PATH + "/")
    and path.lower().endswith(".csv")
]

print("   Phase 2 CSV files found :", len(phase2_files))

if len(phase2_files) != 10:
    raise RuntimeError(
        f"Expected 10 Phase 2 CSV files, "
        f"but found {len(phase2_files)}."
    )

print("   Phase 2 directory       : VERIFIED ✅")

# ----------------------------------------------------------------------
# Display Phase 2 files
# ----------------------------------------------------------------------

print()
print("4. Phase 2 input files:")

for i, path in enumerate(sorted(phase2_files), start=1):
    print(f"   {i:02d}. {path}")

print()
print("=" * 70)
print("✅ STEP 3.2 COMPLETE")
print("=" * 70)

PHASE 3 — HUGGING FACE CONNECTION

1. Checking Hugging Face authentication...
   Authentication : SUCCESSFUL ✅
   Account        : PrakulShetty

2. Checking project repository...
   Repository     : PrakulShetty/NetVision-AI-SIH26153
   Repository     : ACCESSIBLE ✅
   Total files    : 21

3. Checking Phase 2 dataset directory...
   Phase 2 CSV files found : 10
   Phase 2 directory       : VERIFIED ✅

4. Phase 2 input files:
   01. processed/phase2/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
   02. processed/phase2/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
   03. processed/phase2/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
   04. processed/phase2/Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
   05. processed/phase2/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
   06. processed/phase2/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
   07. processed/phase2/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
   08. processed/phase2/Wednesday-14-02-2018_TrafficForML_C

## 3.3 Phase 2 Input Dataset Verification

This section verifies the Phase 2 datasets that will serve as the input
for feature engineering.

The verification checks:

- Presence of all 10 Phase 2 CSV files
- Expected row count for each file
- Required `Timestamp` and `Label` columns
- CIC-IDS2018 network traffic feature schema
- Additional columns present in the Tuesday dataset
- Overall row count consistency

No data is modified during this step.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.3
# PHASE 2 INPUT DATASET VERIFICATION
# ======================================================================

import os
import pandas as pd
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------
# Expected Phase 2 files and verified final row counts
# ----------------------------------------------------------------------

EXPECTED_FILES = {
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv": 1_043_116,
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv": 900_988,
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_960,
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv": 7_948_746,
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv": 331_027,
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv": 1_046_154,
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_288,
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982,
}

EXPECTED_TOTAL_ROWS = 15_822_221

# ----------------------------------------------------------------------
# Required common columns
# ----------------------------------------------------------------------

EXPECTED_COMMON_COLUMNS = [
    "Dst Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "TotLen Fwd Pkts",
    "TotLen Bwd Pkts",
    "Fwd Pkt Len Max",
    "Fwd Pkt Len Min",
    "Fwd Pkt Len Mean",
    "Fwd Pkt Len Std",
    "Bwd Pkt Len Max",
    "Bwd Pkt Len Min",
    "Bwd Pkt Len Mean",
    "Bwd Pkt Len Std",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Tot",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Tot",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "Fwd Header Len",
    "Bwd Header Len",
    "Fwd Pkts/s",
    "Bwd Pkts/s",
    "Pkt Len Min",
    "Pkt Len Max",
    "Pkt Len Mean",
    "Pkt Len Std",
    "Pkt Len Var",
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
    "Down/Up Ratio",
    "Pkt Size Avg",
    "Fwd Seg Size Avg",
    "Bwd Seg Size Avg",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
    "Subflow Fwd Pkts",
    "Subflow Fwd Byts",
    "Subflow Bwd Pkts",
    "Subflow Bwd Byts",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Act Data Pkts",
    "Fwd Seg Size Min",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",
    "Label",
]

# ----------------------------------------------------------------------
# Start verification
# ----------------------------------------------------------------------

print("=" * 70)
print("PHASE 3 — STEP 3.3")
print("PHASE 2 INPUT DATASET VERIFICATION")
print("=" * 70)
print()

verification_results = []
total_rows = 0

# ----------------------------------------------------------------------
# Verify each file
# ----------------------------------------------------------------------

for index, (filename, expected_rows) in enumerate(
    EXPECTED_FILES.items(), start=1
):

    print(f"[{index}/10] {filename}")

    remote_path = f"{PHASE2_PATH}/{filename}"

    # Download only to Colab cache.
    # The original file in Hugging Face is NOT modified.
    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=remote_path,
        repo_type="dataset"
    )

    # --------------------------------------------------------------
    # Read header only
    # --------------------------------------------------------------

    header_df = pd.read_csv(
        local_file,
        nrows=0
    )

    columns = list(header_df.columns)

    # --------------------------------------------------------------
    # Verify required columns
    # --------------------------------------------------------------

    missing_columns = [
        col for col in EXPECTED_COMMON_COLUMNS
        if col not in columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing required columns in {filename}: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------------
    # Count rows without loading the entire CSV into memory
    # --------------------------------------------------------------

    with open(local_file, "rb") as f:
        actual_rows = sum(1 for _ in f) - 1

    # --------------------------------------------------------------
    # Identify extra columns
    # --------------------------------------------------------------

    extra_columns = [
        col
        for col in columns
        if col not in EXPECTED_COMMON_COLUMNS
    ]

    row_match = actual_rows == expected_rows

    print(f"    Expected rows : {expected_rows:,}")
    print(f"    Actual rows   : {actual_rows:,}")
    print(f"    Columns       : {len(columns)}")

    if row_match:
        print("    Row count     : VERIFIED ✅")
    else:
        print("    Row count     : MISMATCH ❌")
        raise RuntimeError(
            f"Row count mismatch in {filename}. "
            f"Expected {expected_rows:,}, got {actual_rows:,}."
        )

    if extra_columns:
        print(f"    Extra columns : {extra_columns}")

    print()

    verification_results.append({
        "File": filename,
        "Expected Rows": expected_rows,
        "Actual Rows": actual_rows,
        "Columns": len(columns),
        "Extra Columns": ", ".join(extra_columns)
            if extra_columns else "None",
        "Status": "VERIFIED"
    })

    total_rows += actual_rows

# ----------------------------------------------------------------------
# Final verification
# ----------------------------------------------------------------------

results_df = pd.DataFrame(verification_results)

print("=" * 70)
print("FINAL PHASE 2 INPUT VERIFICATION")
print("=" * 70)

print(f"Files verified      : {len(results_df)}/10")
print(f"Expected total rows: {EXPECTED_TOTAL_ROWS:,}")
print(f"Actual total rows  : {total_rows:,}")

if total_rows != EXPECTED_TOTAL_ROWS:
    raise RuntimeError(
        f"TOTAL ROW COUNT MISMATCH! "
        f"Expected {EXPECTED_TOTAL_ROWS:,}, "
        f"got {total_rows:,}."
    )

print("Total row count     : VERIFIED ✅")
print("Required schema     : VERIFIED ✅")
print()
print("Phase 2 input is ready for feature engineering. ✅")

display(results_df)

print()
print("=" * 70)
print("✅ STEP 3.3 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.3
PHASE 2 INPUT DATASET VERIFICATION

[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


processed/phase2/Friday-02-03-2018_Traff(…): reconstructing file:   0%|          |  0.00B /  351MB            

processed/phase2/Friday-02-03-2018_Traff(…): downloading bytes:           |  0.00B            

    Expected rows : 1,043,116
    Actual rows   : 1,043,116
    Columns       : 80
    Row count     : VERIFIED ✅

[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


processed/phase2/Friday-16-02-2018_Traff(…): reconstructing file:   0%|          |  0.00B /  300MB            

processed/phase2/Friday-16-02-2018_Traff(…): downloading bytes:           |  0.00B            

... (23 outputs left out)

processed/phase2/Wednesday-28-02-2018_Tr(…): downloading bytes:           |  0.00B            

    Expected rows : 606,982
    Actual rows   : 606,982
    Columns       : 81
    Row count     : VERIFIED ✅
    Extra columns : ['Timestamp_1']

FINAL PHASE 2 INPUT VERIFICATION
Files verified      : 10/10
Expected total rows: 15,822,221
Actual total rows  : 15,822,221
Total row count     : VERIFIED ✅
Required schema     : VERIFIED ✅

Phase 2 input is ready for feature engineering. ✅


,File,Expected Rows,Actual Rows,Columns,Extra Columns,Status
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,1043116,80,None,VERIFIED
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,900988,80,None,VERIFIED
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,1045960,80,None,VERIFIED
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,7948746,84,"Flow ID, Src IP, Src Port, Dst IP",VERIFIED
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,331027,80,None,VERIFIED
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,1046154,80,None,VERIFIED
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,1045288,80,None,VERIFIED
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,822942,81,Timestamp_1,VERIFIED
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,1031018,81,Timestamp_1,VERIFIED
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,606982,81,Timestamp_1,VERIFIED



✅ STEP 3.3 COMPLETE


In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.3
# SCHEMA ANOMALY INVESTIGATION
# ======================================================================

import os
import pandas as pd
from huggingface_hub import hf_hub_download

AFFECTED_FILES = [
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv",
]

print("=" * 70)
print("PHASE 3 — STEP 3.3")
print("SCHEMA ANOMALY INVESTIGATION")
print("=" * 70)
print()

for filename in AFFECTED_FILES:

    print("-" * 70)
    print(filename)
    print("-" * 70)

    remote_path = f"{PHASE2_PATH}/{filename}"

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=remote_path,
        repo_type="dataset"
    )

    # Read only the first few rows
    sample = pd.read_csv(
        local_file,
        nrows=10
    )

    print("Column count:", len(sample.columns))

    print("\nTimestamp-related columns:")
    timestamp_columns = [
        col for col in sample.columns
        if "Timestamp" in col
    ]

    for col in timestamp_columns:
        print(f"   {col}")

    # --------------------------------------------------------------
    # Compare Timestamp and Timestamp_1 if both exist
    # --------------------------------------------------------------

    if "Timestamp" in sample.columns and "Timestamp_1" in sample.columns:

        print("\nFirst 5 Timestamp values:")
        print(sample["Timestamp"].head().to_string(index=False))

        print("\nFirst 5 Timestamp_1 values:")
        print(sample["Timestamp_1"].head().to_string(index=False))

        timestamp_equal = (
            sample["Timestamp"].astype(str)
            == sample["Timestamp_1"].astype(str)
        )

        print(
            "\nTimestamp == Timestamp_1 for sample rows:",
            timestamp_equal.tolist()
        )

        print(
            "All sample values identical:",
            timestamp_equal.all()
        )

    else:
        print("\nTimestamp_1 is not present in this sample.")

    print()

print("=" * 70)
print("✅ SCHEMA ANOMALY INVESTIGATION COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.3
SCHEMA ANOMALY INVESTIGATION

----------------------------------------------------------------------
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
----------------------------------------------------------------------
Column count: 81

Timestamp-related columns:
   Timestamp
   Timestamp_1

First 5 Timestamp values:
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00

First 5 Timestamp_1 values:
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00
14/02/2018 01:00:00

Timestamp == Timestamp_1 for sample rows: [True, True, True, True, True, True, True, True, True, True]
All sample values identical: True

----------------------------------------------------------------------
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
----------------------------------------------------------------------
Column count: 81

Timestamp-related columns:
   Timestamp
   Timestamp_1

First 5 Timestamp value

### 3.3.1 Schema Correction — Duplicate Timestamp Column

During Phase 3 input verification, three recovered Phase 2 files were found
to contain an additional `Timestamp_1` column.

Investigation confirmed that `Timestamp_1` is identical to the original
`Timestamp` column. Therefore, `Timestamp_1` is a duplicate recovery artifact
and does not represent an additional network-traffic feature.

Only this duplicate column is removed. Row counts and all other columns are
preserved.

Affected files:

- Wednesday-14-02-2018
- Wednesday-21-02-2018
- Wednesday-28-02-2018

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.3.1
# SCHEMA CORRECTION
# Remove duplicate Timestamp_1 from affected Phase 2 files
# ======================================================================

import os
import shutil
import duckdb
from huggingface_hub import hf_hub_download, HfApi

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------

AFFECTED_FILES = {
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982,
}

PHASE2_PATH = "processed/phase2"

WORK_DIR = "/content/phase3_schema_correction"
os.makedirs(WORK_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# DuckDB configuration
# ----------------------------------------------------------------------

con = duckdb.connect()

con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
con.execute("SET preserve_insertion_order=false")

TEMP_DIR = "/content/duckdb_temp_schema_correction"
os.makedirs(TEMP_DIR, exist_ok=True)
con.execute(f"SET temp_directory='{TEMP_DIR}'")

api = HfApi()

# ----------------------------------------------------------------------
# Start
# ----------------------------------------------------------------------

print("=" * 70)
print("PHASE 3 — STEP 3.3.1")
print("SCHEMA CORRECTION")
print("=" * 70)
print()

for index, (filename, expected_rows) in enumerate(
    AFFECTED_FILES.items(), start=1
):

    print(f"[{index}/3] {filename}")
    print("-" * 70)

    remote_path = f"{PHASE2_PATH}/{filename}"

    # --------------------------------------------------------------
    # Download current Phase 2 file
    # --------------------------------------------------------------

    print("1. Accessing Phase 2 file...")

    source_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=remote_path,
        repo_type="dataset"
    )

    print("   File downloaded successfully.")

    # --------------------------------------------------------------
    # Inspect schema
    # --------------------------------------------------------------

    print("2. Checking schema...")

    schema_df = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            '{source_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false,
            parallel=false
        )
        """
    ).fetchdf()

    columns = schema_df["column_name"].tolist()

    if "Timestamp_1" not in columns:
        raise RuntimeError(
            f"Timestamp_1 was not found in {filename}. "
            "Correction cancelled."
        )

    if "Timestamp" not in columns:
        raise RuntimeError(
            f"Timestamp column missing in {filename}. "
            "Correction cancelled."
        )

    print(f"   Current columns: {len(columns)}")
    print("   Timestamp       : PRESENT ✅")
    print("   Timestamp_1     : PRESENT ✅")

    # --------------------------------------------------------------
    # Verify Timestamp and Timestamp_1 are identical
    # --------------------------------------------------------------

    print("3. Verifying Timestamp duplication...")

    comparison = con.execute(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT_IF(
                COALESCE("Timestamp", '') !=
                COALESCE("Timestamp_1", '')
            ) AS mismatched_rows
        FROM read_csv_auto(
            '{source_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false,
            parallel=false
        )
        """
    ).fetchone()

    total_rows_before = comparison[0]
    mismatched_rows = comparison[1]

    print(f"   Rows checked    : {total_rows_before:,}")
    print(f"   Mismatched rows : {mismatched_rows:,}")

    if total_rows_before != expected_rows:
        raise RuntimeError(
            f"Unexpected row count before correction in {filename}. "
            f"Expected {expected_rows:,}, "
            f"got {total_rows_before:,}."
        )

    if mismatched_rows != 0:
        raise RuntimeError(
            f"Timestamp and Timestamp_1 are NOT identical in {filename}. "
            "Correction cancelled."
        )

    print("   Timestamp duplication VERIFIED ✅")

    # --------------------------------------------------------------
    # Create corrected file
    # --------------------------------------------------------------

    output_file = os.path.join(WORK_DIR, filename)

    print("4. Removing only Timestamp_1...")

    query = f"""
    COPY (
        SELECT * EXCLUDE ("Timestamp_1")
        FROM read_csv_auto(
            '{source_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false,
            parallel=false
        )
    )
    TO '{output_file}'
    (
        FORMAT CSV,
        HEADER TRUE,
        DELIMITER ','
    )
    """

    con.execute(query)

    # --------------------------------------------------------------
    # Verify corrected file
    # --------------------------------------------------------------

    print("5. Verifying corrected file...")

    corrected_schema = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            '{output_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false,
            parallel=false
        )
        """
    ).fetchdf()

    corrected_columns = corrected_schema["column_name"].tolist()

    corrected_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{output_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false,
            parallel=false
        )
        """
    ).fetchone()[0]

    print(f"   Expected rows : {expected_rows:,}")
    print(f"   Actual rows   : {corrected_rows:,}")
    print(f"   Columns       : {len(corrected_columns)}")

    if corrected_rows != expected_rows:
        raise RuntimeError(
            f"Row count changed during correction for {filename}."
        )

    if len(corrected_columns) != 80:
        raise RuntimeError(
            f"Expected 80 columns after correction, "
            f"got {len(corrected_columns)}."
        )

    if "Timestamp" not in corrected_columns:
        raise RuntimeError(
            f"Timestamp missing after correction for {filename}."
        )

    if "Timestamp_1" in corrected_columns:
        raise RuntimeError(
            f"Timestamp_1 still exists after correction for {filename}."
        )

    print("   Row count       : VERIFIED ✅")
    print("   80-column schema: VERIFIED ✅")
    print("   Timestamp       : PRESENT ✅")
    print("   Timestamp_1     : REMOVED ✅")

    # --------------------------------------------------------------
    # Upload corrected file
    # --------------------------------------------------------------

    print("6. Uploading corrected Phase 2 file...")

    api.upload_file(
        path_or_fileobj=output_file,
        path_in_repo=remote_path,
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=f"fix: remove duplicate Timestamp_1 from {filename}"
    )

    print("   Upload successful. ✅")

    # --------------------------------------------------------------
    # Verify remote file exists
    # --------------------------------------------------------------

    print("7. Verifying remote file...")

    remote_files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="dataset"
    )

    if remote_path not in remote_files:
        raise RuntimeError(
            f"Remote verification failed for {filename}."
        )

    print("   Remote file VERIFIED ✅")

    # --------------------------------------------------------------
    # Remove local corrected copy
    # --------------------------------------------------------------

    if os.path.exists(output_file):
        os.remove(output_file)

    print()
    print(f"✅ {filename} CORRECTED AND SECURED")
    print()

# ----------------------------------------------------------------------
# Cleanup
# ----------------------------------------------------------------------

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR, ignore_errors=True)

con.close()

print("=" * 70)
print("✅ STEP 3.3.1 COMPLETE")
print("=" * 70)
print()
print("Three affected Phase 2 files were corrected.")
print("Only duplicate Timestamp_1 columns were removed.")
print("All expected row counts were preserved.")
print("All corrected files were uploaded and remotely verified.")

PHASE 3 — STEP 3.3.1
SCHEMA CORRECTION

[1/3] Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
----------------------------------------------------------------------
1. Accessing Phase 2 file...
   File downloaded successfully.
2. Checking schema...
   Current columns: 81
   Timestamp       : PRESENT ✅
   Timestamp_1     : PRESENT ✅
3. Verifying Timestamp duplication...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Rows checked    : 822,942
   Mismatched rows : 0
   Timestamp duplication VERIFIED ✅
4. Removing only Timestamp_1...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5. Verifying corrected file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (17 outputs left out)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ficForML_CICFlowMeter.csv:   1%|          | 1.38MB /  208MB            

   Upload successful. ✅
7. Verifying remote file...
   Remote file VERIFIED ✅

✅ Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv CORRECTED AND SECURED

✅ STEP 3.3.1 COMPLETE

Three affected Phase 2 files were corrected.
Only duplicate Timestamp_1 columns were removed.
All expected row counts were preserved.
All corrected files were uploaded and remotely verified.


### 3.3.2 Final Phase 2 Input Verification

The Phase 2 input datasets were previously verified for row counts and schema.
During verification, three recovered files were found to contain a duplicate
`Timestamp_1` column. The duplicate column has now been removed.

This section performs the final verification of all 10 Phase 2 files before
feature engineering begins.

The Phase 2 dataset will be treated as read-only after this verification.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.3.2
# CORRECTED FINAL PHASE 2 INPUT VERIFICATION
# ======================================================================

import os
import pandas as pd
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------
# Expected Phase 2 files and final row counts
# ----------------------------------------------------------------------

EXPECTED_FILES = {
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv": 1_043_116,
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv": 900_988,
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_960,
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv": 7_948_746,
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv": 331_027,
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv": 1_046_154,
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_288,
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982,
}

EXPECTED_TOTAL_ROWS = 15_822_221

# ----------------------------------------------------------------------
# Common 80-column ML schema
# ----------------------------------------------------------------------

COMMON_COLUMNS = [
    "Dst Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "TotLen Fwd Pkts",
    "TotLen Bwd Pkts",
    "Fwd Pkt Len Max",
    "Fwd Pkt Len Min",
    "Fwd Pkt Len Mean",
    "Fwd Pkt Len Std",
    "Bwd Pkt Len Max",
    "Bwd Pkt Len Min",
    "Bwd Pkt Len Mean",
    "Bwd Pkt Len Std",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Tot",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Tot",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "Fwd Header Len",
    "Bwd Header Len",
    "Fwd Pkts/s",
    "Bwd Pkts/s",
    "Pkt Len Min",
    "Pkt Len Max",
    "Pkt Len Mean",
    "Pkt Len Std",
    "Pkt Len Var",
    "FIN Flag Cnt",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "PSH Flag Cnt",
    "ACK Flag Cnt",
    "URG Flag Cnt",
    "CWE Flag Count",
    "ECE Flag Cnt",
    "Down/Up Ratio",
    "Pkt Size Avg",
    "Fwd Seg Size Avg",
    "Bwd Seg Size Avg",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
    "Subflow Fwd Pkts",
    "Subflow Fwd Byts",
    "Subflow Bwd Pkts",
    "Subflow Bwd Byts",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "Fwd Act Data Pkts",
    "Fwd Seg Size Min",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",
    "Label",
]

# ----------------------------------------------------------------------
# Legitimate extra columns found only in Tuesday file
# ----------------------------------------------------------------------

TUESDAY_EXTRA_COLUMNS = [
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
]

# ----------------------------------------------------------------------
# Verification
# ----------------------------------------------------------------------

print("=" * 70)
print("PHASE 3 — STEP 3.3.2")
print("CORRECTED FINAL PHASE 2 INPUT VERIFICATION")
print("=" * 70)
print()

results = []
total_rows = 0

for index, (filename, expected_rows) in enumerate(
    EXPECTED_FILES.items(), start=1
):

    print(f"[{index}/10] {filename}")

    remote_path = f"{PHASE2_PATH}/{filename}"

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=remote_path,
        repo_type="dataset"
    )

    # --------------------------------------------------------------
    # Read header only
    # --------------------------------------------------------------

    header = pd.read_csv(
        local_file,
        nrows=0
    )

    columns = list(header.columns)

    # --------------------------------------------------------------
    # Check common ML columns
    # --------------------------------------------------------------

    missing_columns = [
        col for col in COMMON_COLUMNS
        if col not in columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing common ML columns in {filename}: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------------
    # Identify extra columns
    # --------------------------------------------------------------

    extra_columns = [
        col for col in columns
        if col not in COMMON_COLUMNS
    ]

    # --------------------------------------------------------------
    # Validate extra columns
    # --------------------------------------------------------------

    if filename == "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv":

        if set(extra_columns) != set(TUESDAY_EXTRA_COLUMNS):
            raise RuntimeError(
                f"Unexpected extra columns in Tuesday file: "
                f"{extra_columns}"
            )

        schema_status = "80 common + 4 legitimate extra"

    else:

        if extra_columns:
            raise RuntimeError(
                f"Unexpected extra columns in {filename}: "
                f"{extra_columns}"
            )

        schema_status = "80 common"

    # --------------------------------------------------------------
    # Row count
    # --------------------------------------------------------------

    with open(local_file, "rb") as f:
        actual_rows = sum(1 for _ in f) - 1

    if actual_rows != expected_rows:
        raise RuntimeError(
            f"Row count mismatch in {filename}. "
            f"Expected {expected_rows:,}, "
            f"got {actual_rows:,}."
        )

    total_rows += actual_rows

    print(f"    Rows    : {actual_rows:,} ✅")
    print(f"    Columns : {len(columns)}")
    print(f"    Schema  : {schema_status} ✅")
    print()

    results.append({
        "File": filename,
        "Rows": actual_rows,
        "Total Columns": len(columns),
        "Common ML Columns": len(COMMON_COLUMNS),
        "Extra Columns": len(extra_columns),
        "Extra Column Names": ", ".join(extra_columns) if extra_columns else "-",
        "Status": "READY"
    })

# ----------------------------------------------------------------------
# Final verification
# ----------------------------------------------------------------------

results_df = pd.DataFrame(results)

print("=" * 70)
print("FINAL PHASE 2 INPUT STATUS")
print("=" * 70)

print(f"Files verified       : {len(results_df)}/10")
print(f"Total rows           : {total_rows:,}")
print(f"Expected total rows  : {EXPECTED_TOTAL_ROWS:,}")
print(f"Common ML columns    : {len(COMMON_COLUMNS)}")
print()

# --------------------------------------------------------------
# Final assertions
# --------------------------------------------------------------

if len(results_df) != 10:
    raise RuntimeError("FILE COUNT VERIFICATION FAILED.")

if total_rows != EXPECTED_TOTAL_ROWS:
    raise RuntimeError("TOTAL ROW COUNT VERIFICATION FAILED.")

if not all(
    results_df["Common ML Columns"] == len(COMMON_COLUMNS)
):
    raise RuntimeError("COMMON COLUMN VERIFICATION FAILED.")

# Tuesday must have 84 columns
tuesday_result = results_df[
    results_df["File"].str.startswith("Thuesday-20-02-2018")
]

if len(tuesday_result) != 1:
    raise RuntimeError("Tuesday file verification failed.")

if tuesday_result.iloc[0]["Total Columns"] != 84:
    raise RuntimeError(
        "Tuesday file should contain 84 total columns."
    )

# All other files must have 80 columns
other_results = results_df[
    ~results_df["File"].str.startswith("Thuesday-20-02-2018")
]

if not all(other_results["Total Columns"] == 80):
    raise RuntimeError(
        "One or more non-Tuesday files do not have 80 columns."
    )

print("10/10 files verified                 ✅")
print("15,822,221 total rows verified       ✅")
print("80 common ML columns verified        ✅")
print("Tuesday 84-column schema verified   ✅")
print("No unexpected columns               ✅")
print()
print("PHASE 2 INPUT DATASET: VERIFIED ✅")
print("PHASE 2 DATASET IS NOW READ-ONLY 🔒")
print()
print("=" * 70)
print("✅ STEP 3.3.2 COMPLETE")
print("=" * 70)

display(results_df)

PHASE 3 — STEP 3.3.2
CORRECTED FINAL PHASE 2 INPUT VERIFICATION

[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 1,043,116 ✅
    Columns : 80
    Schema  : 80 common ✅

[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 900,988 ✅
    Columns : 80
    Schema  : 80 common ✅

[3/10] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 1,045,960 ✅
    Columns : 80
    Schema  : 80 common ✅

[4/10] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 7,948,746 ✅
    Columns : 84
    Schema  : 80 common + 4 legitimate extra ✅

[5/10] Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 331,027 ✅
    Columns : 80
    Schema  : 80 common ✅

[6/10] Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 1,046,154 ✅
    Columns : 80
    Schema  : 80 common ✅

[7/10] Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
    Rows    : 1,045,288 ✅
    Columns : 80
    Schema  : 80 common ✅

[8/10] Wednesday-14-02-2018_Traffic

processed/phase2/Wednesday-14-02-2018_Tr(…): reconstructing file:   0%|          |  0.00B /  308MB            

processed/phase2/Wednesday-14-02-2018_Tr(…): downloading bytes:           |  0.00B            

    Rows    : 822,942 ✅
    Columns : 80
    Schema  : 80 common ✅

[9/10] Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv


processed/phase2/Wednesday-21-02-2018_Tr(…): reconstructing file:   0%|          |  0.00B /  326MB            

processed/phase2/Wednesday-21-02-2018_Tr(…): downloading bytes:           |  0.00B            

    Rows    : 1,031,018 ✅
    Columns : 80
    Schema  : 80 common ✅

[10/10] Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


processed/phase2/Wednesday-28-02-2018_Tr(…): reconstructing file:   0%|          |  0.00B /  208MB            

processed/phase2/Wednesday-28-02-2018_Tr(…): downloading bytes:           |  0.00B            

    Rows    : 606,982 ✅
    Columns : 80
    Schema  : 80 common ✅

FINAL PHASE 2 INPUT STATUS
Files verified       : 10/10
Total rows           : 15,822,221
Expected total rows  : 15,822,221
Common ML columns    : 80

10/10 files verified                 ✅
15,822,221 total rows verified       ✅
80 common ML columns verified        ✅
Tuesday 84-column schema verified   ✅
No unexpected columns               ✅

PHASE 2 INPUT DATASET: VERIFIED ✅
PHASE 2 DATASET IS NOW READ-ONLY 🔒

✅ STEP 3.3.2 COMPLETE


,File,Rows,Total Columns,Common ML Columns,Extra Columns,Extra Column Names,Status
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,80,80,0,-,READY
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,80,80,0,-,READY
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,80,80,0,-,READY
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,84,80,4,"Flow ID, Src IP, Src Port, Dst IP",READY
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,80,80,0,-,READY
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,80,80,0,-,READY
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,80,80,0,-,READY
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,80,80,0,-,READY
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,80,80,0,-,READY
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,80,80,0,-,READY


### 3.4.1 Numeric Feature Type Investigation

The Phase 2 datasets contain network-flow features generated from CICFlowMeter.

Before creating derived features, the numerical representation of the dataset
must be investigated.

This step identifies numeric, categorical, and label columns across the
10 Phase 2 files without modifying the source datasets.

The purpose is to establish a consistent feature-processing schema for
subsequent feature engineering.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.1
# NUMERIC FEATURE TYPE INVESTIGATION
# ======================================================================

import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.4.1")
print("NUMERIC FEATURE TYPE INVESTIGATION")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Files to investigate
# ----------------------------------------------------------------------

files_to_check = list(EXPECTED_FILES.keys())

# ----------------------------------------------------------------------
# Store observations
# ----------------------------------------------------------------------

file_summaries = []

all_columns = set()
numeric_columns_by_file = {}
non_numeric_columns_by_file = {}

# ----------------------------------------------------------------------
# Inspect each Phase 2 file
# ----------------------------------------------------------------------

for index, filename in enumerate(files_to_check, start=1):

    print(f"[{index}/10] Inspecting: {filename}")

    remote_path = f"{PHASE2_PATH}/{filename}"

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=remote_path,
        repo_type="dataset"
    )

    # --------------------------------------------------------------
    # Read a representative sample
    # --------------------------------------------------------------

    sample = pd.read_csv(
        local_file,
        nrows=10000,
        low_memory=False
    )

    # --------------------------------------------------------------
    # Record columns
    # --------------------------------------------------------------

    columns = list(sample.columns)
    all_columns.update(columns)

    # --------------------------------------------------------------
    # Identify numeric columns
    # --------------------------------------------------------------

    numeric_cols = sample.select_dtypes(
        include=np.number
    ).columns.tolist()

    non_numeric_cols = [
        col for col in columns
        if col not in numeric_cols
    ]

    numeric_columns_by_file[filename] = numeric_cols
    non_numeric_columns_by_file[filename] = non_numeric_cols

    # --------------------------------------------------------------
    # Summary
    # --------------------------------------------------------------

    file_summaries.append({
        "File": filename,
        "Columns": len(columns),
        "Numeric Columns": len(numeric_cols),
        "Non-Numeric Columns": len(non_numeric_cols),
        "Rows Sampled": len(sample)
    })

    print(f"    Total columns    : {len(columns)}")
    print(f"    Numeric columns  : {len(numeric_cols)}")
    print(f"    Non-numeric      : {len(non_numeric_cols)}")
    print()

# ----------------------------------------------------------------------
# Display summary
# ----------------------------------------------------------------------

summary_df = pd.DataFrame(file_summaries)

print("=" * 70)
print("NUMERIC FEATURE INVESTIGATION SUMMARY")
print("=" * 70)

display(summary_df)

# ----------------------------------------------------------------------
# Compare numeric schemas across files
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("NUMERIC SCHEMA CONSISTENCY")
print("=" * 70)

reference_file = files_to_check[0]

reference_numeric = set(
    numeric_columns_by_file[reference_file]
)

schema_consistent = True

for filename in files_to_check[1:]:

    current_numeric = set(
        numeric_columns_by_file[filename]
    )

    missing = reference_numeric - current_numeric
    additional = current_numeric - reference_numeric

    if missing or additional:

        schema_consistent = False

        print()
        print(f"Schema difference: {filename}")

        if missing:
            print("    Missing numeric columns:", sorted(missing))

        if additional:
            print("    Additional numeric columns:", sorted(additional))

# ----------------------------------------------------------------------
# Show non-numeric columns
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("NON-NUMERIC COLUMNS")
print("=" * 70)

for filename in files_to_check:

    print()
    print(filename)

    for col in non_numeric_columns_by_file[filename]:
        print(f"    - {col}")

# ----------------------------------------------------------------------
# Final result
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 3.4.1 RESULT")
print("=" * 70)

print(f"Files investigated       : {len(files_to_check)}/10")
print(f"Unique columns observed  : {len(all_columns)}")
print(f"Numeric schema consistent: {schema_consistent}")

print()
print("No source dataset has been modified.")
print("This step only investigates feature data types.")
print()
print("=" * 70)
print("✅ STEP 3.4.1 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.4.1
NUMERIC FEATURE TYPE INVESTIGATION

[1/10] Inspecting: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 80
    Numeric columns  : 78
    Non-numeric      : 2

[2/10] Inspecting: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 80
    Numeric columns  : 78
    Non-numeric      : 2

[3/10] Inspecting: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 80
    Numeric columns  : 78
    Non-numeric      : 2

[4/10] Inspecting: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 84
    Numeric columns  : 79
    Non-numeric      : 5

[5/10] Inspecting: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 80
    Numeric columns  : 78
    Non-numeric      : 2

[6/10] Inspecting: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
    Total columns    : 80
    Numeric columns  : 78
    Non-numeric      : 2

[7/10] Inspecting: Thursday-22-02-2018_TrafficForML_CICFlowMeter.cs

,File,Columns,Numeric Columns,Non-Numeric Columns,Rows Sampled
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,84,79,5,10000
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,10000
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,80,78,2,10000
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,80,78,2,10000
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,80,78,2,10000



NUMERIC SCHEMA CONSISTENCY

Schema difference: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
    Additional numeric columns: ['Src Port']

NON-NUMERIC COLUMNS

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
    - Flow ID
    - Src IP
    - Dst IP
    - Timestamp
    - Label

Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
    - Timestamp
    - Label

STEP 3.4

### 3.4.2 Feature Category Definition

The Phase 2 investigation identified temporal, target, network identity,
and numerical network-flow fields.

This step formally defines these feature categories so that subsequent
feature engineering uses a consistent representation across all 10 files.

The original Phase 2 datasets are not modified.

Feature categories:

1. Temporal field — Timestamp
2. Target field — Label
3. Tuesday-only identity fields — Flow ID, Src IP, Src Port, Dst IP
4. Common network-flow features — the remaining 80 common fields

Timestamp and Label will be handled separately from numerical feature
engineering. The common ML feature space will be maintained consistently
across all files.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.2
# FEATURE CATEGORY DEFINITION
# ======================================================================

print("=" * 70)
print("PHASE 3 — STEP 3.4.2")
print("FEATURE CATEGORY DEFINITION")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Define important column groups
# ----------------------------------------------------------------------

TEMPORAL_COLUMNS = [
    "Timestamp"
]

TARGET_COLUMNS = [
    "Label"
]

IDENTITY_COLUMNS = [
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP"
]

# ----------------------------------------------------------------------
# Common ML columns
# ----------------------------------------------------------------------

COMMON_ML_COLUMNS = COMMON_COLUMNS.copy()

# Remove temporal and target fields from model input features
MODEL_FEATURE_COLUMNS = [
    col
    for col in COMMON_ML_COLUMNS
    if col not in TEMPORAL_COLUMNS + TARGET_COLUMNS
]

# ----------------------------------------------------------------------
# Numeric model features
# ----------------------------------------------------------------------

NUMERIC_MODEL_FEATURES = [
    col
    for col in MODEL_FEATURE_COLUMNS
    if col != "Protocol"
]

# Protocol and ports are numeric representations but are treated as
# categorical/network protocol identifiers during later analysis.
IDENTIFIER_LIKE_NUMERIC = [
    "Dst Port",
    "Protocol"
]

# ----------------------------------------------------------------------
# Display category sizes
# ----------------------------------------------------------------------

print("FEATURE CATEGORIES")
print("-" * 70)

print(f"Common ML columns          : {len(COMMON_ML_COLUMNS)}")
print(f"Temporal columns           : {len(TEMPORAL_COLUMNS)}")
print(f"Target columns             : {len(TARGET_COLUMNS)}")
print(f"Model input columns        : {len(MODEL_FEATURE_COLUMNS)}")
print(f"Identity columns available : {len(IDENTITY_COLUMNS)}")

print()

print("Temporal columns:")
for col in TEMPORAL_COLUMNS:
    print(f"  - {col}")

print()

print("Target columns:")
for col in TARGET_COLUMNS:
    print(f"  - {col}")

print()

print("Tuesday-only identity columns:")
for col in IDENTITY_COLUMNS:
    print(f"  - {col}")

print()

print("Identifier-like numeric columns:")
for col in IDENTIFIER_LIKE_NUMERIC:
    print(f"  - {col}")

print()

# ----------------------------------------------------------------------
# Validate category definitions
# ----------------------------------------------------------------------

assert "Timestamp" in COMMON_ML_COLUMNS
assert "Label" in COMMON_ML_COLUMNS

assert "Timestamp" not in MODEL_FEATURE_COLUMNS
assert "Label" not in MODEL_FEATURE_COLUMNS

assert len(COMMON_ML_COLUMNS) == 80
assert len(MODEL_FEATURE_COLUMNS) == 78

print("=" * 70)
print("CATEGORY VALIDATION")
print("=" * 70)

print("80 common ML columns             : VERIFIED ✅")
print("Timestamp separated              : VERIFIED ✅")
print("Label separated                  : VERIFIED ✅")
print("78 model-input columns           : VERIFIED ✅")
print("Tuesday identity fields isolated : VERIFIED ✅")

print()
print("No source dataset has been modified.")
print()
print("=" * 70)
print("✅ STEP 3.4.2 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.4.2
FEATURE CATEGORY DEFINITION

FEATURE CATEGORIES
----------------------------------------------------------------------
Common ML columns          : 80
Temporal columns           : 1
Target columns             : 1
Model input columns        : 78
Identity columns available : 4

Temporal columns:
  - Timestamp

Target columns:
  - Label

Tuesday-only identity columns:
  - Flow ID
  - Src IP
  - Src Port
  - Dst IP

Identifier-like numeric columns:
  - Dst Port
  - Protocol

CATEGORY VALIDATION
80 common ML columns             : VERIFIED ✅
Timestamp separated              : VERIFIED ✅
Label separated                  : VERIFIED ✅
78 model-input columns           : VERIFIED ✅
Tuesday identity fields isolated : VERIFIED ✅

No source dataset has been modified.

✅ STEP 3.4.2 COMPLETE


### 3.4.3 Numeric Value Quality Investigation

This step investigates the numerical behavior of the common model-input
features across all 10 Phase 2 datasets.

The investigation checks:

- Numeric conversion validity
- Minimum and maximum values
- Negative values
- Infinite values

Because the Tuesday file contains four legitimate additional columns,
the files are analyzed independently rather than combined with UNION ALL.

The Phase 2 source datasets are not modified.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.3
# OPTIMIZED NUMERIC VALUE QUALITY INVESTIGATION
# ======================================================================

import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.4.3")
print("OPTIMIZED NUMERIC VALUE QUALITY INVESTIGATION")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Common model-input features
# ----------------------------------------------------------------------

NUMERIC_FEATURES = MODEL_FEATURE_COLUMNS.copy()

print(f"Model input features investigated : {len(NUMERIC_FEATURES)}")
print()

# ----------------------------------------------------------------------
# DuckDB connection
# ----------------------------------------------------------------------

con = duckdb.connect()

all_results = []

# ----------------------------------------------------------------------
# Process each file independently
# ----------------------------------------------------------------------

for file_index, filename in enumerate(
    EXPECTED_FILES.keys(),
    start=1
):

    print(f"[{file_index}/10] Investigating: {filename}")

    # --------------------------------------------------------------
    # Download / access Phase 2 file
    # --------------------------------------------------------------

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"{PHASE2_PATH}/{filename}",
        repo_type="dataset"
    )

    escaped_path = local_file.replace("'", "''")

    # --------------------------------------------------------------
    # Get available common model features
    # --------------------------------------------------------------

    header = pd.read_csv(
        local_file,
        nrows=0
    )

    available_features = [
        col
        for col in NUMERIC_FEATURES
        if col in header.columns
    ]

    if len(available_features) != len(NUMERIC_FEATURES):
        missing = [
            col
            for col in NUMERIC_FEATURES
            if col not in available_features
        ]

        raise RuntimeError(
            f"Missing model features in {filename}: {missing}"
        )

    # --------------------------------------------------------------
    # Build ONE aggregation query for ALL features
    # --------------------------------------------------------------

    aggregation_expressions = []

    for col in available_features:

        quoted_col = '"' + col.replace('"', '""') + '"'

        # Valid numeric count
        aggregation_expressions.append(
            f"""
            COUNT(
                TRY_CAST({quoted_col} AS DOUBLE)
            ) AS "{col}__valid"
            """
        )

        # Minimum
        aggregation_expressions.append(
            f"""
            MIN(
                TRY_CAST({quoted_col} AS DOUBLE)
            ) AS "{col}__min"
            """
        )

        # Maximum
        aggregation_expressions.append(
            f"""
            MAX(
                TRY_CAST({quoted_col} AS DOUBLE)
            ) AS "{col}__max"
            """
        )

        # Negative count
        aggregation_expressions.append(
            f"""
            SUM(
                CASE
                    WHEN TRY_CAST({quoted_col} AS DOUBLE) < 0
                    THEN 1
                    ELSE 0
                END
            ) AS "{col}__negative"
            """
        )

        # Infinite count
        aggregation_expressions.append(
            f"""
            SUM(
                CASE
                    WHEN LOWER(TRIM({quoted_col}))
                    IN ('inf', '+inf', '-inf')
                    THEN 1
                    ELSE 0
                END
            ) AS "{col}__infinite"
            """
        )

    aggregation_sql = ",\n".join(
        aggregation_expressions
    )

    # --------------------------------------------------------------
    # ONE full scan of the file
    # --------------------------------------------------------------

    query = f"""
        SELECT
            COUNT(*) AS total_rows,
            {aggregation_sql}

        FROM read_csv(
            '{escaped_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false
        )
    """

    result = con.execute(query).fetchone()

    column_names = [
        desc[0]
        for desc in con.description
    ]

    result_dict = dict(
        zip(column_names, result)
    )

    total_rows = result_dict["total_rows"]

    # --------------------------------------------------------------
    # Convert one-file result into feature-level rows
    # --------------------------------------------------------------

    for col in available_features:

        valid_numeric = result_dict[f"{col}__valid"]
        min_value = result_dict[f"{col}__min"]
        max_value = result_dict[f"{col}__min"]  # corrected below

        # Retrieve maximum separately
        max_value = result_dict[f"{col}__max"]

        negative_values = result_dict[
            f"{col}__negative"
        ]

        infinite_values = result_dict[
            f"{col}__infinite"
        ]

        invalid_numeric = (
            total_rows - valid_numeric
        )

        all_results.append({
            "File": filename,
            "Feature": col,
            "Total Rows": total_rows,
            "Valid Numeric": valid_numeric,
            "Invalid Numeric": invalid_numeric,
            "Min": min_value,
            "Max": max_value,
            "Negative Values": negative_values,
            "Infinite Values": infinite_values
        })

    print(
        f"    Rows scanned     : {total_rows:,}"
    )

    print(
        f"    Features checked : {len(available_features)}"
    )

    print(
        "    Single-pass scan : COMPLETE ✅"
    )

    print()

# ----------------------------------------------------------------------
# Convert results to DataFrame
# ----------------------------------------------------------------------

numeric_quality_df = pd.DataFrame(
    all_results
)

# ----------------------------------------------------------------------
# Calculate percentages
# ----------------------------------------------------------------------

numeric_quality_df["Invalid %"] = (
    numeric_quality_df["Invalid Numeric"]
    / numeric_quality_df["Total Rows"]
    * 100
)

numeric_quality_df["Negative %"] = (
    numeric_quality_df["Negative Values"]
    / numeric_quality_df["Total Rows"]
    * 100
)

numeric_quality_df["Infinite %"] = (
    numeric_quality_df["Infinite Values"]
    / numeric_quality_df["Total Rows"]
    * 100
)

# ----------------------------------------------------------------------
# Aggregate results across all 10 files
# ----------------------------------------------------------------------

aggregated_quality_df = (
    numeric_quality_df
    .groupby("Feature", as_index=False)
    .agg({
        "Total Rows": "sum",
        "Valid Numeric": "sum",
        "Invalid Numeric": "sum",
        "Negative Values": "sum",
        "Infinite Values": "sum"
    })
)

# ----------------------------------------------------------------------
# Recalculate global percentages
# ----------------------------------------------------------------------

aggregated_quality_df["Invalid %"] = (
    aggregated_quality_df["Invalid Numeric"]
    / aggregated_quality_df["Total Rows"]
    * 100
)

aggregated_quality_df["Negative %"] = (
    aggregated_quality_df["Negative Values"]
    / aggregated_quality_df["Total Rows"]
    * 100
)

aggregated_quality_df["Infinite %"] = (
    aggregated_quality_df["Infinite Values"]
    / aggregated_quality_df["Total Rows"]
    * 100
)

# ----------------------------------------------------------------------
# Global minimum and maximum
# ----------------------------------------------------------------------

range_df = (
    numeric_quality_df
    .groupby("Feature", as_index=False)
    .agg(
        Min=("Min", "min"),
        Max=("Max", "max")
    )
)

aggregated_quality_df = aggregated_quality_df.merge(
    range_df,
    on="Feature",
    how="left"
)

# ----------------------------------------------------------------------
# Final column order
# ----------------------------------------------------------------------

aggregated_quality_df = aggregated_quality_df[
    [
        "Feature",
        "Total Rows",
        "Valid Numeric",
        "Invalid Numeric",
        "Invalid %",
        "Min",
        "Max",
        "Negative Values",
        "Negative %",
        "Infinite Values",
        "Infinite %"
    ]
]

# ----------------------------------------------------------------------
# Display complete quality summary
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("NUMERIC FEATURE QUALITY SUMMARY")
print("=" * 70)

display(
    aggregated_quality_df
)

# ----------------------------------------------------------------------
# Features requiring interpretation
# ----------------------------------------------------------------------

problem_features_df = aggregated_quality_df[
    (aggregated_quality_df["Invalid Numeric"] > 0) |
    (aggregated_quality_df["Negative Values"] > 0) |
    (aggregated_quality_df["Infinite Values"] > 0)
]

print()
print("=" * 70)
print("FEATURES REQUIRING INTERPRETATION")
print("=" * 70)

if problem_features_df.empty:

    print(
        "No features require additional numerical interpretation. ✅"
    )

else:

    print(
        f"{len(problem_features_df)} feature(s) "
        "contain values requiring interpretation:"
    )

    display(
        problem_features_df[
            [
                "Feature",
                "Min",
                "Max",
                "Invalid Numeric",
                "Negative Values",
                "Infinite Values"
            ]
        ]
    )

# ----------------------------------------------------------------------
# Final checks
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("DATASET-WIDE NUMERIC CHECKS")
print("=" * 70)

expected_rows = sum(
    EXPECTED_FILES.values()
)

print(
    f"Expected total rows              : "
    f"{expected_rows:,}"
)

print(
    f"Features investigated            : "
    f"{len(aggregated_quality_df)}"
)

print(
    f"Features with invalid values     : "
    f"{(aggregated_quality_df['Invalid Numeric'] > 0).sum()}"
)

print(
    f"Features with negative values    : "
    f"{(aggregated_quality_df['Negative Values'] > 0).sum()}"
)

print(
    f"Features with infinite values    : "
    f"{(aggregated_quality_df['Infinite Values'] > 0).sum()}"
)

print()
print("No source dataset has been modified.")
print()
print("=" * 70)
print("✅ STEP 3.4.3 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.4.3
OPTIMIZED NUMERIC VALUE QUALITY INVESTIGATION

Model input features investigated : 78

[1/10] Investigating: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Rows scanned     : 1,043,116
    Features checked : 78
    Single-pass scan : COMPLETE ✅

[2/10] Investigating: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Rows scanned     : 900,988
    Features checked : 78
    Single-pass scan : COMPLETE ✅

[3/10] Investigating: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (15 outputs left out)

,Feature,Total Rows,Valid Numeric,Invalid Numeric,Invalid %,Min,Max,Negative Values,Negative %,Infinite Values,Infinite %
0,ACK Flag Cnt,15822221,15822221,0,0.0,0.0,1.000000e+00,0,0.0,0,0.0
1,Active Max,15822221,15822221,0,0.0,0.0,1.140000e+08,0,0.0,0,0.0
2,Active Mean,15822221,15822221,0,0.0,0.0,1.140000e+08,0,0.0,0,0.0
3,Active Min,15822221,15822221,0,0.0,0.0,1.140000e+08,0,0.0,0,0.0
4,Active Std,15822221,15822221,0,0.0,0.0,7.523241e+07,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
73,Tot Bwd Pkts,15822221,15822221,0,0.0,0.0,1.231180e+05,0,0.0,0,0.0
74,Tot Fwd Pkts,15822221,15822221,0,0.0,1.0,3.096290e+05,0,0.0,0,0.0
75,TotLen Bwd Pkts,15822221,15822221,0,0.0,0.0,1.563604e+08,0,0.0,0,0.0
76,TotLen Fwd Pkts,15822221,15822221,0,0.0,0.0,1.443918e+08,0,0.0,0,0.0



FEATURES REQUIRING INTERPRETATION
4 feature(s) contain values requiring interpretation:


,Feature,Min,Max,Invalid Numeric,Negative Values,Infinite Values
27,Flow Byts/s,0.000000,2077429.984,36005,0,0
33,Flow Pkts/s,0.016667,6000000.000,94348,0,0
57,Init Bwd Win Byts,-1.000000,65535.000,0,8192276,0
58,Init Fwd Win Byts,-1.000000,65535.000,0,4432096,0



DATASET-WIDE NUMERIC CHECKS
Expected total rows              : 15,822,221
Features investigated            : 78
Features with invalid values     : 2
Features with negative values    : 2
Features with infinite values    : 0

No source dataset has been modified.

✅ STEP 3.4.3 COMPLETE


### 3.4.4 Anomalous Feature Interpretation

Step 3.4.3 identified four numerical features requiring further
interpretation:

- Flow Byts/s
- Flow Pkts/s
- Init Bwd Win Byts
- Init Fwd Win Byts

This step investigates the actual values associated with these
features and their relationship with protocol information.

The purpose is to distinguish genuinely invalid values from
dataset-specific special values before deciding on any transformation.

No Phase 2 source dataset is modified in this step.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.4
# ANOMALOUS FEATURE INTERPRETATION
# ======================================================================

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.4.4")
print("ANOMALOUS FEATURE INTERPRETATION")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Features under investigation
# ----------------------------------------------------------------------

ANOMALOUS_FEATURES = [
    "Flow Byts/s",
    "Flow Pkts/s",
    "Init Bwd Win Byts",
    "Init Fwd Win Byts"
]

print("Features under investigation:")
for feature in ANOMALOUS_FEATURES:
    print(f"  - {feature}")

print()

# ----------------------------------------------------------------------
# DuckDB connection
# ----------------------------------------------------------------------

con = duckdb.connect()

# ----------------------------------------------------------------------
# Results containers
# ----------------------------------------------------------------------

invalid_value_results = []
negative_window_results = []

# ----------------------------------------------------------------------
# Process each file independently
# ----------------------------------------------------------------------

for file_index, filename in enumerate(
    EXPECTED_FILES.keys(),
    start=1
):

    print(f"[{file_index}/10] Investigating: {filename}")

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"{PHASE2_PATH}/{filename}",
        repo_type="dataset"
    )

    escaped_path = local_file.replace("'", "''")

    # --------------------------------------------------------------
    # Check Flow Byts/s and Flow Pkts/s invalid values
    # --------------------------------------------------------------

    query_invalid = f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN TRY_CAST("Flow Byts/s" AS DOUBLE) IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS flow_bytes_invalid,

            SUM(
                CASE
                    WHEN TRY_CAST("Flow Pkts/s" AS DOUBLE) IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS flow_packets_invalid,

            SUM(
                CASE
                    WHEN LOWER(TRIM("Flow Byts/s"))
                    IN ('nan', 'na', 'n/a', '')
                    THEN 1
                    ELSE 0
                END
            ) AS flow_bytes_nan_like,

            SUM(
                CASE
                    WHEN LOWER(TRIM("Flow Pkts/s"))
                    IN ('nan', 'na', 'n/a', '')
                    THEN 1
                    ELSE 0
                END
            ) AS flow_packets_nan_like

        FROM read_csv(
            '{escaped_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false
        )
    """

    invalid_result = con.execute(
        query_invalid
    ).fetchone()

    invalid_value_results.append({
        "File": filename,
        "Total Rows": invalid_result[0],
        "Flow Byts/s Invalid": invalid_result[1],
        "Flow Pkts/s Invalid": invalid_result[2],
        "Flow Byts/s NaN-like": invalid_result[3],
        "Flow Pkts/s NaN-like": invalid_result[4]
    })

    # --------------------------------------------------------------
    # Investigate -1 values in initial TCP window features
    # --------------------------------------------------------------

    query_windows = f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Bwd Win Byts" AS DOUBLE) = -1
                    THEN 1
                    ELSE 0
                END
            ) AS bwd_minus_one,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Fwd Win Byts" AS DOUBLE) = -1
                    THEN 1
                    ELSE 0
                END
            ) AS fwd_minus_one,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Bwd Win Byts" AS DOUBLE) >= 0
                    THEN 1
                    ELSE 0
                END
            ) AS bwd_nonnegative,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Fwd Win Byts" AS DOUBLE) >= 0
                    THEN 1
                    ELSE 0
                END
            ) AS fwd_nonnegative

        FROM read_csv(
            '{escaped_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false
        )
    """

    window_result = con.execute(
        query_windows
    ).fetchone()

    negative_window_results.append({
        "File": filename,
        "Total Rows": window_result[0],
        "Init Bwd Win = -1": window_result[1],
        "Init Fwd Win = -1": window_result[2],
        "Init Bwd Win >= 0": window_result[3],
        "Init Fwd Win >= 0": window_result[4]
    })

    print("    Investigation complete ✅")

# ----------------------------------------------------------------------
# Convert to DataFrames
# ----------------------------------------------------------------------

invalid_df = pd.DataFrame(
    invalid_value_results
)

window_df = pd.DataFrame(
    negative_window_results
)

# ----------------------------------------------------------------------
# Display invalid Flow rate results
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("FLOW RATE INVALID-VALUE INVESTIGATION")
print("=" * 70)

display(invalid_df)

# ----------------------------------------------------------------------
# Display window-size -1 investigation
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("INITIAL WINDOW SIZE -1 INVESTIGATION")
print("=" * 70)

display(window_df)

# ----------------------------------------------------------------------
# Aggregate totals
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("DATASET-WIDE RESULTS")
print("=" * 70)

print(
    f"Flow Byts/s invalid values : "
    f"{invalid_df['Flow Byts/s Invalid'].sum():,}"
)

print(
    f"Flow Pkts/s invalid values : "
    f"{invalid_df['Flow Pkts/s Invalid'].sum():,}"
)

print(
    f"Flow Byts/s NaN-like values: "
    f"{invalid_df['Flow Byts/s NaN-like'].sum():,}"
)

print(
    f"Flow Pkts/s NaN-like values: "
    f"{invalid_df['Flow Pkts/s NaN-like'].sum():,}"
)

print()

print(
    f"Init Bwd Win = -1          : "
    f"{window_df['Init Bwd Win = -1'].sum():,}"
)

print(
    f"Init Fwd Win = -1          : "
    f"{window_df['Init Fwd Win = -1'].sum():,}"
)

# ----------------------------------------------------------------------
# Final interpretation status
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("INTERPRETATION STATUS")
print("=" * 70)

print(
    "Flow-rate invalid values identified       : "
    f"{invalid_df['Flow Byts/s Invalid'].sum() + invalid_df['Flow Pkts/s Invalid'].sum():,}"
)

print(
    "Initial-window -1 values identified       : "
    f"{window_df['Init Bwd Win = -1'].sum() + window_df['Init Fwd Win = -1'].sum():,}"
)

print()
print("No values have been replaced.")
print("No rows have been deleted.")
print("No Phase 2 source dataset has been modified.")

print()
print("=" * 70)
print("✅ STEP 3.4.4 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.4.4
ANOMALOUS FEATURE INTERPRETATION

Features under investigation:
  - Flow Byts/s
  - Flow Pkts/s
  - Init Bwd Win Byts
  - Init Fwd Win Byts

[1/10] Investigating: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Investigation complete ✅
[2/10] Investigating: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Investigation complete ✅
[3/10] Investigating: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


... (19 outputs left out)

,File,Total Rows,Flow Byts/s Invalid,Flow Pkts/s Invalid,Flow Byts/s NaN-like,Flow Pkts/s NaN-like
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,1486,4044,2558,0
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,0,0,0,0
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,1948,5702,3754,0
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,22686,59452,36766,0
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,1083,2917,1834,0
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,3101,8022,4921,0
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,2040,5604,3564,0
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,1544,3821,2277,0
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,0,0,0,0
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,2117,4786,2669,0



INITIAL WINDOW SIZE -1 INVESTIGATION


,File,Total Rows,Init Bwd Win = -1,Init Fwd Win = -1,Init Bwd Win >= 0,Init Fwd Win >= 0
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,508990,216035,534126,827081
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,411239,125,489749,900863
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,570603,308367,475357,737593
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,4209793,2653104,3738953,5295642
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,159413,118165,171614,212862
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,585864,364014,460290,682140
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,574783,322067,470505,723221
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,346019,219214,476923,603728
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,508437,3707,522581,1027311
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,317135,227298,289847,379684



DATASET-WIDE RESULTS
Flow Byts/s invalid values : 36,005
Flow Pkts/s invalid values : 94,348
Flow Byts/s NaN-like values: 58,343
Flow Pkts/s NaN-like values: 0

Init Bwd Win = -1          : 8,192,276
Init Fwd Win = -1          : 4,432,096

INTERPRETATION STATUS
Flow-rate invalid values identified       : 130,353
Initial-window -1 values identified       : 12,624,372

No values have been replaced.
No rows have been deleted.
No Phase 2 source dataset has been modified.

✅ STEP 3.4.4 COMPLETE


### 3.4.5 Numerical Special-Value Treatment Decision

Based on the file-level investigation:

1. `Flow Byts/s` and `Flow Pkts/s` contain invalid rate values across
   multiple dataset files. These values will be represented as missing
   values (`NaN`) rather than replaced with arbitrary numerical values.

2. `Init Fwd Win Byts` and `Init Bwd Win Byts` contain widespread `-1`
   values across the dataset. These values will be treated as special
   missing-value indicators and converted to `NaN`.

3. Rows will not be removed solely because these feature values are
   missing.

4. The Phase 2 source datasets will remain unchanged. These treatments
   will be applied only when creating the Phase 3 feature-engineered
   datasets.

Further imputation or model-specific handling will be decided after
feature distributions and model requirements are investigated.

### 3.4.5 Numerical Special-Value Treatment Verification

Based on the numerical investigation, the following treatment is proposed:

1. `Flow Byts/s`
   - Invalid/non-numeric values will be represented as `NaN`.

2. `Flow Pkts/s`
   - Invalid/non-numeric values will be represented as `NaN`.

3. `Init Fwd Win Byts`
   - The special value `-1` will be represented as `NaN`.

4. `Init Bwd Win Byts`
   - The special value `-1` will be represented as `NaN`.

This step verifies the number of affected records before applying the
transformation.

No source data is modified.
No rows are removed.
No Phase 3 files are created in this step.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.5
# VERIFY PROPOSED NUMERICAL SPECIAL-VALUE TREATMENT
# ======================================================================

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.4.5")
print("VERIFY PROPOSED NUMERICAL SPECIAL-VALUE TREATMENT")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Features under treatment
# ----------------------------------------------------------------------

TREATMENT_FEATURES = [
    "Flow Byts/s",
    "Flow Pkts/s",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts"
]

# ----------------------------------------------------------------------
# DuckDB connection
# ----------------------------------------------------------------------

con = duckdb.connect()

verification_results = []

# ----------------------------------------------------------------------
# Process each Phase 2 file
# ----------------------------------------------------------------------

for file_index, filename in enumerate(
    EXPECTED_FILES.keys(),
    start=1
):

    print(
        f"[{file_index}/10] Verifying: {filename}"
    )

    local_file = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"{PHASE2_PATH}/{filename}",
        repo_type="dataset"
    )

    escaped_path = local_file.replace("'", "''")

    # --------------------------------------------------------------
    # One scan per file
    # --------------------------------------------------------------

    query = f"""
        SELECT
            COUNT(*) AS total_rows,

            SUM(
                CASE
                    WHEN TRY_CAST("Flow Byts/s" AS DOUBLE) IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS flow_bytes_to_nan,

            SUM(
                CASE
                    WHEN TRY_CAST("Flow Pkts/s" AS DOUBLE) IS NULL
                    THEN 1
                    ELSE 0
                END
            ) AS flow_packets_to_nan,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Fwd Win Byts" AS DOUBLE) = -1
                    THEN 1
                    ELSE 0
                END
            ) AS fwd_window_minus_one,

            SUM(
                CASE
                    WHEN TRY_CAST("Init Bwd Win Byts" AS DOUBLE) = -1
                    THEN 1
                    ELSE 0
                END
            ) AS bwd_window_minus_one

        FROM read_csv(
            '{escaped_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=false
        )
    """

    result = con.execute(query).fetchone()

    total_rows = result[0]
    flow_bytes = result[1]
    flow_packets = result[2]
    fwd_window = result[3]
    bwd_window = result[4]

    verification_results.append({
        "File": filename,
        "Rows": total_rows,
        "Flow Byts/s → NaN": flow_bytes,
        "Flow Pkts/s → NaN": flow_packets,
        "Init Fwd Win -1 → NaN": fwd_window,
        "Init Bwd Win -1 → NaN": bwd_window
    })

    print(
        f"    Rows checked : {total_rows:,} ✅"
    )

    print(
        "    Treatment counts verified ✅"
    )

# ----------------------------------------------------------------------
# Results DataFrame
# ----------------------------------------------------------------------

treatment_df = pd.DataFrame(
    verification_results
)

# ----------------------------------------------------------------------
# Dataset-wide totals
# ----------------------------------------------------------------------

total_flow_bytes = treatment_df[
    "Flow Byts/s → NaN"
].sum()

total_flow_packets = treatment_df[
    "Flow Pkts/s → NaN"
].sum()

total_fwd_window = treatment_df[
    "Init Fwd Win -1 → NaN"
].sum()

total_bwd_window = treatment_df[
    "Init Bwd Win -1 → NaN"
].sum()

# ----------------------------------------------------------------------
# Display file-level verification
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("FILE-LEVEL TREATMENT COUNTS")
print("=" * 70)

display(treatment_df)

# ----------------------------------------------------------------------
# Display global totals
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("DATASET-WIDE TREATMENT COUNTS")
print("=" * 70)

print(
    f"Flow Byts/s invalid → NaN      : "
    f"{total_flow_bytes:,}"
)

print(
    f"Flow Pkts/s invalid → NaN      : "
    f"{total_flow_packets:,}"
)

print(
    f"Init Fwd Win -1 → NaN          : "
    f"{total_fwd_window:,}"
)

print(
    f"Init Bwd Win -1 → NaN          : "
    f"{total_bwd_window:,}"
)

print()

total_values_affected = (
    total_flow_bytes
    + total_flow_packets
    + total_fwd_window
    + total_bwd_window
)

print(
    f"Total feature-values affected  : "
    f"{total_values_affected:,}"
)

# ----------------------------------------------------------------------
# Expected values from Step 3.4.4
# ----------------------------------------------------------------------

EXPECTED_FLOW_BYTES = 36_005
EXPECTED_FLOW_PACKETS = 94_348
EXPECTED_FWD_WINDOW = 4_432_096
EXPECTED_BWD_WINDOW = 8_192_276

# ----------------------------------------------------------------------
# Verify consistency with previous investigation
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("CONSISTENCY CHECK WITH STEP 3.4.4")
print("=" * 70)

assert total_flow_bytes == EXPECTED_FLOW_BYTES
assert total_flow_packets == EXPECTED_FLOW_PACKETS
assert total_fwd_window == EXPECTED_FWD_WINDOW
assert total_bwd_window == EXPECTED_BWD_WINDOW

print("Flow Byts/s count     : MATCHED ✅")
print("Flow Pkts/s count     : MATCHED ✅")
print("Init Fwd Win -1 count : MATCHED ✅")
print("Init Bwd Win -1 count : MATCHED ✅")

# ----------------------------------------------------------------------
# Verify row counts
# ----------------------------------------------------------------------

expected_total_rows = sum(
    EXPECTED_FILES.values()
)

actual_total_rows = treatment_df[
    "Rows"
].sum()

assert actual_total_rows == expected_total_rows

print()
print(
    f"Total rows verified   : "
    f"{actual_total_rows:,} ✅"
)

# ----------------------------------------------------------------------
# Final status
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("TREATMENT DECISION STATUS")
print("=" * 70)

print("Special-value counts verified        : YES ✅")
print("Row counts preserved                 : YES ✅")
print("Phase 2 source modified              : NO ✅")
print("Transformation applied               : NO")
print("Phase 3 output created               : NO")

print()
print(
    "The proposed numerical treatment is now "
    "verified and ready for application."
)

print()
print("=" * 70)
print("✅ STEP 3.4.5 COMPLETE")
print("=" * 70)

PHASE 3 — STEP 3.4.5
VERIFY PROPOSED NUMERICAL SPECIAL-VALUE TREATMENT

[1/10] Verifying: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Rows checked : 1,043,116 ✅
    Treatment counts verified ✅
[2/10] Verifying: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Rows checked : 900,988 ✅
    Treatment counts verified ✅
[3/10] Verifying: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (9 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Rows checked : 1,031,018 ✅
    Treatment counts verified ✅
[10/10] Verifying: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
    Rows checked : 606,982 ✅
    Treatment counts verified ✅

FILE-LEVEL TREATMENT COUNTS


,File,Rows,Flow Byts/s → NaN,Flow Pkts/s → NaN,Init Fwd Win -1 → NaN,Init Bwd Win -1 → NaN
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,1486,4044,216035,508990
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,0,0,125,411239
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,1948,5702,308367,570603
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,22686,59452,2653104,4209793
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,1083,2917,118165,159413
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,3101,8022,364014,585864
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,2040,5604,322067,574783
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,1544,3821,219214,346019
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,0,0,3707,508437
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,2117,4786,227298,317135



DATASET-WIDE TREATMENT COUNTS
Flow Byts/s invalid → NaN      : 36,005
Flow Pkts/s invalid → NaN      : 94,348
Init Fwd Win -1 → NaN          : 4,432,096
Init Bwd Win -1 → NaN          : 8,192,276

Total feature-values affected  : 12,754,725

CONSISTENCY CHECK WITH STEP 3.4.4
Flow Byts/s count     : MATCHED ✅
Flow Pkts/s count     : MATCHED ✅
Init Fwd Win -1 count : MATCHED ✅
Init Bwd Win -1 count : MATCHED ✅

Total rows verified   : 15,822,221 ✅

TREATMENT DECISION STATUS
Special-value counts verified        : YES ✅
Row counts preserved                 : YES ✅
Phase 2 source modified              : NO ✅
Transformation applied               : NO
Phase 3 output created               : NO

The proposed numerical treatment is now verified and ready for application.

✅ STEP 3.4.5 COMPLETE


### 3.4.6 Apply Numerical Special-Value Treatment

This step applies the numerical treatment verified in Step 3.4.5.

The following transformations are applied:

1. Flow Byts/s invalid values → NaN
2. Flow Pkts/s invalid values → NaN
3. Init Fwd Win Byts: -1 → NaN
4. Init Bwd Win Byts: -1 → NaN

No rows are removed.

The Phase 2 datasets are used as read-only input.
The transformed datasets are saved separately under:

processed/phase3/

This preserves the dataset lineage:

Phase 2 → Phase 3

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.6
# APPLY NUMERICAL TREATMENT & CREATE PHASE 3 DATASETS
# ======================================================================

import os
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, HfApi

print("=" * 70)
print("PHASE 3 — STEP 3.4.6")
print("APPLY NUMERICAL TREATMENT & CREATE PHASE 3 DATASETS")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# HUGGING FACE PROJECT CONFIGURATION
# ----------------------------------------------------------------------

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"

PHASE2_PATH = "processed/phase2"
PHASE3_PATH = "processed/phase3"

LOCAL_PHASE3_DIR = "/content/phase3_working"

os.makedirs(LOCAL_PHASE3_DIR, exist_ok=True)

print(f"HF Repository : {HF_REPO}")
print(f"Phase 2 path  : {PHASE2_PATH}")
print(f"Phase 3 path  : {PHASE3_PATH}")
print()

# ----------------------------------------------------------------------
# EXPECTED TREATMENT COUNTS
# Verified in Step 3.4.5
# ----------------------------------------------------------------------

EXPECTED_TREATMENT_COUNTS = {
    "Flow Byts/s": 36_005,
    "Flow Pkts/s": 94_348,
    "Init Fwd Win Byts": 4_432_096,
    "Init Bwd Win Byts": 8_192_276
}

# ----------------------------------------------------------------------
# EXPECTED PHASE 2 ROW COUNTS
# ----------------------------------------------------------------------

EXPECTED_ROWS = {
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv": 1_043_116,
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv": 900_988,
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_960,
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv": 7_948_746,
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv": 331_027,
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv": 1_046_154,
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_288,
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982
}

# ----------------------------------------------------------------------
# HUGGING FACE API
# ----------------------------------------------------------------------

api = HfApi()

repo_info = api.repo_info(
    repo_id=HF_REPO,
    repo_type="dataset"
)

print("Hugging Face repository access: VERIFIED ✅")
print()

# ----------------------------------------------------------------------
# PROCESS EACH PHASE 2 FILE
# ----------------------------------------------------------------------

phase3_results = []

for file_index, (filename, expected_rows) in enumerate(
    EXPECTED_ROWS.items(),
    start=1
):

    print("=" * 70)
    print(f"[{file_index}/10] Processing:")
    print(filename)
    print("=" * 70)

    # --------------------------------------------------------------
    # Download Phase 2 input
    # --------------------------------------------------------------

    phase2_local = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"{PHASE2_PATH}/{filename}",
        repo_type="dataset"
    )

    print("Phase 2 input downloaded ✅")

    # --------------------------------------------------------------
    # Read Phase 2 dataset
    # --------------------------------------------------------------

    df = pd.read_csv(
        phase2_local,
        low_memory=False
    )

    original_rows = len(df)

    print(f"Input rows: {original_rows:,}")

    # --------------------------------------------------------------
    # Verify Phase 2 row count
    # --------------------------------------------------------------

    if original_rows != expected_rows:
        raise RuntimeError(
            f"Row count mismatch in {filename}: "
            f"expected {expected_rows:,}, "
            f"got {original_rows:,}"
        )

    # --------------------------------------------------------------
    # Required columns
    # --------------------------------------------------------------

    required_columns = [
        "Flow Byts/s",
        "Flow Pkts/s",
        "Init Fwd Win Byts",
        "Init Bwd Win Byts",
        "Timestamp",
        "Label"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing required columns in {filename}: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------------
    # Count values BEFORE treatment
    # --------------------------------------------------------------

    flow_bytes = pd.to_numeric(
        df["Flow Byts/s"],
        errors="coerce"
    )

    flow_packets = pd.to_numeric(
        df["Flow Pkts/s"],
        errors="coerce"
    )

    fwd_window = pd.to_numeric(
        df["Init Fwd Win Byts"],
        errors="coerce"
    )

    bwd_window = pd.to_numeric(
        df["Init Bwd Win Byts"],
        errors="coerce"
    )

    flow_bytes_invalid = int(
        flow_bytes.isna().sum()
    )

    flow_packets_invalid = int(
        flow_packets.isna().sum()
    )

    fwd_minus_one = int(
        (fwd_window == -1).sum()
    )

    bwd_minus_one = int(
        (bwd_window == -1).sum()
    )

    # --------------------------------------------------------------
    # APPLY VERIFIED TREATMENT
    # --------------------------------------------------------------

    df["Flow Byts/s"] = flow_bytes

    df["Flow Pkts/s"] = flow_packets

    df["Init Fwd Win Byts"] = (
        fwd_window.replace(-1, np.nan)
    )

    df["Init Bwd Win Byts"] = (
        bwd_window.replace(-1, np.nan)
    )

    # --------------------------------------------------------------
    # Verify row count preservation
    # --------------------------------------------------------------

    final_rows = len(df)

    if final_rows != original_rows:
        raise RuntimeError(
            f"Rows changed in {filename}: "
            f"{original_rows:,} → {final_rows:,}"
        )

    # --------------------------------------------------------------
    # Verify -1 values were removed
    # --------------------------------------------------------------

    remaining_fwd_minus_one = int(
        (df["Init Fwd Win Byts"] == -1).sum()
    )

    remaining_bwd_minus_one = int(
        (df["Init Bwd Win Byts"] == -1).sum()
    )

    if remaining_fwd_minus_one != 0:
        raise RuntimeError(
            f"Init Fwd Win Byts still contains -1 "
            f"in {filename}"
        )

    if remaining_bwd_minus_one != 0:
        raise RuntimeError(
            f"Init Bwd Win Byts still contains -1 "
            f"in {filename}"
        )

    # --------------------------------------------------------------
    # Save Phase 3 file locally
    # --------------------------------------------------------------

    local_output = os.path.join(
        LOCAL_PHASE3_DIR,
        filename
    )

    df.to_csv(
        local_output,
        index=False
    )

    print("Phase 3 file created locally ✅")

    # --------------------------------------------------------------
    # Upload Phase 3 file to Hugging Face
    # --------------------------------------------------------------

    api.upload_file(
        path_or_fileobj=local_output,
        path_in_repo=f"{PHASE3_PATH}/{filename}",
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=(
            f"Create Phase 3 processed dataset: {filename}"
        )
    )

    print("Phase 3 file uploaded to Hugging Face ✅")

    # --------------------------------------------------------------
    # Store results
    # --------------------------------------------------------------

    phase3_results.append({
        "File": filename,
        "Rows": final_rows,
        "Flow Byts/s → NaN": flow_bytes_invalid,
        "Flow Pkts/s → NaN": flow_packets_invalid,
        "Init Fwd Win -1 → NaN": fwd_minus_one,
        "Init Bwd Win -1 → NaN": bwd_minus_one,
        "Status": "CREATED & UPLOADED"
    })

    print(f"Final rows: {final_rows:,} ✅")
    print()

    # --------------------------------------------------------------
    # Release memory before next large file
    # --------------------------------------------------------------

    del df
    del flow_bytes
    del flow_packets
    del fwd_window
    del bwd_window

# ----------------------------------------------------------------------
# CREATE SUMMARY
# ----------------------------------------------------------------------

phase3_df = pd.DataFrame(
    phase3_results
)

print("=" * 70)
print("PHASE 3 CREATION SUMMARY")
print("=" * 70)

display(phase3_df)

# ----------------------------------------------------------------------
# DATASET-WIDE TREATMENT TOTALS
# ----------------------------------------------------------------------

actual_totals = {
    "Flow Byts/s": int(
        phase3_df["Flow Byts/s → NaN"].sum()
    ),
    "Flow Pkts/s": int(
        phase3_df["Flow Pkts/s → NaN"].sum()
    ),
    "Init Fwd Win Byts": int(
        phase3_df["Init Fwd Win -1 → NaN"].sum()
    ),
    "Init Bwd Win Byts": int(
        phase3_df["Init Bwd Win -1 → NaN"].sum()
    )
}

print()
print("=" * 70)
print("TREATMENT COUNT VALIDATION")
print("=" * 70)

for feature, expected_count in EXPECTED_TREATMENT_COUNTS.items():

    actual_count = actual_totals[feature]

    print(
        f"{feature:22s}: "
        f"{actual_count:,} / {expected_count:,}"
    )

    if actual_count != expected_count:
        raise RuntimeError(
            f"Treatment count mismatch for {feature}: "
            f"expected {expected_count:,}, "
            f"got {actual_count:,}"
        )

    print(" " * 24 + "MATCHED ✅")

# ----------------------------------------------------------------------
# TOTAL ROW VALIDATION
# ----------------------------------------------------------------------

expected_total_rows = sum(
    EXPECTED_ROWS.values()
)

actual_total_rows = int(
    phase3_df["Rows"].sum()
)

print()
print("=" * 70)
print("TOTAL ROW VALIDATION")
print("=" * 70)

print(
    f"Expected total rows : {expected_total_rows:,}"
)

print(
    f"Phase 3 total rows  : {actual_total_rows:,}"
)

if actual_total_rows != expected_total_rows:
    raise RuntimeError(
        "Phase 3 total row count mismatch."
    )

print("Total rows preserved : YES ✅")

# ----------------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 3.4.6 FINAL STATUS")
print("=" * 70)

print("10/10 Phase 2 files processed       ✅")
print("10/10 Phase 3 files uploaded        ✅")
print("15,822,221 rows preserved           ✅")
print("Four numerical treatments applied   ✅")
print("Treatment counts matched            ✅")
print("Phase 2 datasets preserved          ✅")
print()
print("PHASE 3 DATASET CREATION: COMPLETE ✅")

print()
print("=" * 70)
print("NEXT STEP: 3.4.7 VALIDATION")
print("=" * 70)

PHASE 3 — STEP 3.4.6
APPLY NUMERICAL TREATMENT & CREATE PHASE 3 DATASETS

HF Repository : PrakulShetty/NetVision-AI-SIH26153
Phase 2 path  : processed/phase2
Phase 3 path  : processed/phase3

Hugging Face repository access: VERIFIED ✅

[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Phase 2 input downloaded ✅
Input rows: 1,043,116
Phase 3 file created locally ✅


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ficForML_CICFlowMeter.csv:   2%|1         | 7.97MB /  406MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Phase 3 file uploaded to Hugging Face ✅
Final rows: 1,043,116 ✅

[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Phase 2 input downloaded ✅
Input rows: 900,988
Phase 3 file created locally ✅


... (6 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...ficForML_CICFlowMeter.csv:   4%|3         | 16.0MB /  411MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Phase 3 file uploaded to Hugging Face ✅
Final rows: 1,045,960 ✅

[4/10] Processing:
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Phase 2 input downloaded ✅


In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.4.6 RECOVERY
# CONTINUE FROM FILE 4
# APPLY NUMERICAL TREATMENT USING DUCKDB
# ======================================================================

import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download, HfApi

print("=" * 70)
print("PHASE 3 — STEP 3.4.6 RECOVERY")
print("CONTINUING FROM FILE 4")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# HUGGING FACE CONFIGURATION
# ----------------------------------------------------------------------

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"

PHASE2_PATH = "processed/phase2"
PHASE3_PATH = "processed/phase3"

LOCAL_PHASE3_DIR = "/content/phase3_working"

os.makedirs(LOCAL_PHASE3_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# FILES 4–10 ONLY
# Files 1–3 were already successfully processed.
# ----------------------------------------------------------------------

REMAINING_FILES = {
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv": 7_948_746,
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv": 331_027,
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv": 1_046_154,
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_288,
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982
}

# ----------------------------------------------------------------------
# VERIFIED TREATMENT COUNTS FROM STEP 3.4.5
# Dataset-wide expected totals
# ----------------------------------------------------------------------

EXPECTED_TREATMENT_COUNTS = {
    "Flow Byts/s": 36_005,
    "Flow Pkts/s": 94_348,
    "Init Fwd Win Byts": 4_432_096,
    "Init Bwd Win Byts": 8_192_276
}

# ----------------------------------------------------------------------
# Hugging Face API
# ----------------------------------------------------------------------

api = HfApi()

api.repo_info(
    repo_id=HF_REPO,
    repo_type="dataset"
)

print(f"HF Repository : {HF_REPO}")
print("Repository access : VERIFIED ✅")
print()

# ----------------------------------------------------------------------
# PROCESS FILES 4–10
# ----------------------------------------------------------------------

results = []

for file_number, (filename, expected_rows) in enumerate(
    REMAINING_FILES.items(),
    start=4
):

    print("=" * 70)
    print(f"[{file_number}/10] PROCESSING")
    print(filename)
    print("=" * 70)

    # --------------------------------------------------------------
    # Download Phase 2 file
    # --------------------------------------------------------------

    phase2_local = hf_hub_download(
        repo_id=HF_REPO,
        filename=f"{PHASE2_PATH}/{filename}",
        repo_type="dataset"
    )

    print("Phase 2 input downloaded ✅")

    # --------------------------------------------------------------
    # Local output path
    # --------------------------------------------------------------

    output_file = os.path.join(
        LOCAL_PHASE3_DIR,
        filename
    )

    # --------------------------------------------------------------
    # DuckDB connection
    # --------------------------------------------------------------

    con = duckdb.connect(
        database=":memory:"
    )

    # --------------------------------------------------------------
    # Read CSV through DuckDB
    #
    # Important:
    # DuckDB processes the CSV without loading the complete
    # 7.9 million-row Tuesday file into a Pandas DataFrame.
    # --------------------------------------------------------------

    source = (
        "'" +
        phase2_local.replace("'", "''") +
        "'"
    )

    destination = (
        "'" +
        output_file.replace("'", "''") +
        "'"
    )

    # --------------------------------------------------------------
    # Create transformed Phase 3 CSV
    #
    # TRY_CAST converts invalid numerical representations to NULL.
    # -1 in the initial window features becomes NULL.
    #
    # All other columns are preserved.
    # --------------------------------------------------------------

    query = f"""
    COPY (
        SELECT
            * EXCLUDE (
                "Flow Byts/s",
                "Flow Pkts/s",
                "Init Fwd Win Byts",
                "Init Bwd Win Byts"
            ),

            TRY_CAST("Flow Byts/s" AS DOUBLE)
                AS "Flow Byts/s",

            TRY_CAST("Flow Pkts/s" AS DOUBLE)
                AS "Flow Pkts/s",

            CASE
                WHEN TRY_CAST("Init Fwd Win Byts" AS DOUBLE) = -1
                    THEN NULL
                ELSE TRY_CAST("Init Fwd Win Byts" AS DOUBLE)
            END AS "Init Fwd Win Byts",

            CASE
                WHEN TRY_CAST("Init Bwd Win Byts" AS DOUBLE) = -1
                    THEN NULL
                ELSE TRY_CAST("Init Bwd Win Byts" AS DOUBLE)
            END AS "Init Bwd Win Byts"

        FROM read_csv_auto(
            {source},
            header = TRUE,
            strict_mode = FALSE
        )
    )
    TO {destination}
    WITH (
        FORMAT CSV,
        HEADER TRUE
    )
    """

    print("Applying numerical treatment with DuckDB...")
    print("This may take some time for the Tuesday file.")

    con.execute(query)

    print("Phase 3 file created locally ✅")

    # --------------------------------------------------------------
    # Verify output row count
    # --------------------------------------------------------------

    output_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            {destination},
            header = TRUE,
            strict_mode = FALSE
        )
        """
    ).fetchone()[0]

    output_rows = int(output_rows)

    print(f"Output rows : {output_rows:,}")

    if output_rows != expected_rows:
        con.close()

        raise RuntimeError(
            f"Row count mismatch for {filename}: "
            f"expected {expected_rows:,}, "
            f"got {output_rows:,}"
        )

    print("Row count preserved ✅")

    # --------------------------------------------------------------
    # Verify -1 values are gone
    # --------------------------------------------------------------

    remaining_minus_one = con.execute(
        f"""
        SELECT
            COUNT(*) FILTER (
                WHERE TRY_CAST("Init Fwd Win Byts" AS DOUBLE) = -1
            ) AS fwd_minus_one,

            COUNT(*) FILTER (
                WHERE TRY_CAST("Init Bwd Win Byts" AS DOUBLE) = -1
            ) AS bwd_minus_one
        FROM read_csv_auto(
            {destination},
            header = TRUE,
            strict_mode = FALSE
        )
        """
    ).fetchone()

    fwd_minus_one = int(
        remaining_minus_one[0]
    )

    bwd_minus_one = int(
        remaining_minus_one[1]
    )

    if fwd_minus_one != 0 or bwd_minus_one != 0:
        con.close()

        raise RuntimeError(
            f"-1 values remain in {filename}: "
            f"Fwd={fwd_minus_one:,}, "
            f"Bwd={bwd_minus_one:,}"
        )

    print("Initial-window -1 values removed ✅")

    # --------------------------------------------------------------
    # Count transformed NULL values
    # --------------------------------------------------------------

    null_counts = con.execute(
        f"""
        SELECT
            COUNT(*) FILTER (
                WHERE "Flow Byts/s" IS NULL
            ),

            COUNT(*) FILTER (
                WHERE "Flow Pkts/s" IS NULL
            ),

            COUNT(*) FILTER (
                WHERE "Init Fwd Win Byts" IS NULL
            ),

            COUNT(*) FILTER (
                WHERE "Init Bwd Win Byts" IS NULL
            )

        FROM read_csv_auto(
            {destination},
            header = TRUE,
            strict_mode = FALSE
        )
        """
    ).fetchone()

    flow_bytes_null = int(null_counts[0])
    flow_packets_null = int(null_counts[1])
    fwd_window_null = int(null_counts[2])
    bwd_window_null = int(null_counts[3])

    print(
        f"Flow Byts/s NULL        : {flow_bytes_null:,}"
    )

    print(
        f"Flow Pkts/s NULL        : {flow_packets_null:,}"
    )

    print(
        f"Init Fwd Win Byts NULL  : {fwd_window_null:,}"
    )

    print(
        f"Init Bwd Win Byts NULL  : {bwd_window_null:,}"
    )

    # --------------------------------------------------------------
    # Upload Phase 3 file
    # --------------------------------------------------------------

    print("Uploading Phase 3 file to Hugging Face...")

    api.upload_file(
        path_or_fileobj=output_file,
        path_in_repo=f"{PHASE3_PATH}/{filename}",
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=(
            f"Create Phase 3 processed dataset: {filename}"
        )
    )

    print("Uploaded to Hugging Face ✅")

    # --------------------------------------------------------------
    # Save result
    # --------------------------------------------------------------

    results.append({
        "File": filename,
        "Rows": output_rows,
        "Flow Byts/s NULL": flow_bytes_null,
        "Flow Pkts/s NULL": flow_packets_null,
        "Init Fwd Win Byts NULL": fwd_window_null,
        "Init Bwd Win Byts NULL": bwd_window_null,
        "Status": "COMPLETE"
    })

    con.close()

    print(f"[{file_number}/10] COMPLETE ✅")
    print()

# ----------------------------------------------------------------------
# DISPLAY RESULTS FOR FILES 4–10
# ----------------------------------------------------------------------

results_df = pd.DataFrame(results)

print("=" * 70)
print("FILES 4–10 PROCESSING SUMMARY")
print("=" * 70)

display(results_df)

# ----------------------------------------------------------------------
# Verify total rows for files 4–10
# ----------------------------------------------------------------------

remaining_expected_rows = sum(
    REMAINING_FILES.values()
)

remaining_actual_rows = int(
    results_df["Rows"].sum()
)

print()
print("=" * 70)
print("FILES 4–10 ROW VALIDATION")
print("=" * 70)

print(
    f"Expected rows : {remaining_expected_rows:,}"
)

print(
    f"Actual rows   : {remaining_actual_rows:,}"
)

if remaining_actual_rows != remaining_expected_rows:
    raise RuntimeError(
        "Files 4–10 total row count mismatch."
    )

print("Files 4–10 rows preserved ✅")

# ----------------------------------------------------------------------
# Final status
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 3.4.6 RECOVERY STATUS")
print("=" * 70)

print("Files 1–3 : Already completed ✅")
print("Files 4–10: Processed successfully ✅")
print("DuckDB processing used           ✅")
print("Large Tuesday file handled       ✅")
print("Rows preserved                    ✅")
print("Phase 3 files uploaded           ✅")
print()
print("STEP 3.4.6 COMPLETE FOR ALL 10 FILES ✅")
print()
print("NEXT: STEP 3.4.7 VALIDATION")
print("=" * 70)

PHASE 3 — STEP 3.4.6 RECOVERY
CONTINUING FROM FILE 4

HF Repository : PrakulShetty/NetVision-AI-SIH26153
Repository access : VERIFIED ✅

[4/10] PROCESSING
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Phase 2 input downloaded ✅
Applying numerical treatment with DuckDB...
This may take some time for the Tuesday file.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Phase 3 file created locally ✅


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Output rows : 7,948,746
Row count preserved ✅


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (63 outputs left out)

  ...ficForML_CICFlowMeter.csv:   1%|          | 1.43MB /  225MB            

Uploaded to Hugging Face ✅
[10/10] COMPLETE ✅

FILES 4–10 PROCESSING SUMMARY


,File,Rows,Flow Byts/s NULL,Flow Pkts/s NULL,Init Fwd Win Byts NULL,Init Bwd Win Byts NULL,Status
0,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,22686,59452,2653104,4209793,COMPLETE
1,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,1083,2917,118165,159413,COMPLETE
2,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,3101,8022,364014,585864,COMPLETE
3,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,2040,5604,322067,574783,COMPLETE
4,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,1544,3821,219214,346019,COMPLETE
5,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,0,0,3707,508437,COMPLETE
6,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,2117,4786,227298,317135,COMPLETE



FILES 4–10 ROW VALIDATION
Expected rows : 12,832,157
Actual rows   : 12,832,157
Files 4–10 rows preserved ✅

STEP 3.4.6 RECOVERY STATUS
Files 1–3 : Already completed ✅
Files 4–10: Processed successfully ✅
DuckDB processing used           ✅
Large Tuesday file handled       ✅
Rows preserved                    ✅
Phase 3 files uploaded           ✅

STEP 3.4.6 COMPLETE FOR ALL 10 FILES ✅

NEXT: STEP 3.4.7 VALIDATION


## 3.5 Feature Redundancy Analysis

This step investigates redundancy among the numerical model features.

The analysis uses a representative sample from the Phase 3 datasets
instead of repeatedly scanning all 15.8 million records.

The purpose is to identify highly correlated features that may provide
similar information.

No features are deleted in this step.

The results will be used to guide feature selection before ML modelling.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.5
# FEATURE REDUNDANCY ANALYSIS
# SELF-CONTAINED VERSION
# ======================================================================

import os
import duckdb
import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE 3 — STEP 3.5")
print("FEATURE REDUNDANCY ANALYSIS")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------

PHASE3_DIR = "/content/phase3_working"

SAMPLE_PER_FILE = 10_000

HIGH_CORRELATION_THRESHOLD = 0.95

# ----------------------------------------------------------------------
# EXPECTED FILES
# ----------------------------------------------------------------------

EXPECTED_FILES = [
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv"
]

# ----------------------------------------------------------------------
# VERIFY PHASE 3 FILES
# ----------------------------------------------------------------------

available_files = []

for filename in EXPECTED_FILES:

    file_path = os.path.join(
        PHASE3_DIR,
        filename
    )

    if os.path.exists(file_path):
        available_files.append(file_path)

print(
    f"Phase 3 files available: "
    f"{len(available_files)}/10"
)

if len(available_files) != 10:
    raise RuntimeError(
        "All 10 Phase 3 files must be available locally."
    )

print("All Phase 3 files found ✅")
print()

# ----------------------------------------------------------------------
# DISCOVER SCHEMA FROM FIRST FILE
# ----------------------------------------------------------------------

first_file = available_files[0]

con = duckdb.connect(
    database=":memory:"
)

schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto(
        '{first_file.replace("'", "''")}',
        header = TRUE,
        strict_mode = FALSE
    )
    """
).fetchdf()

con.close()

all_columns = schema_df["column_name"].tolist()

print(
    f"Columns discovered from Phase 3 schema: "
    f"{len(all_columns)}"
)

# ----------------------------------------------------------------------
# DEFINE MODEL FEATURES
# ----------------------------------------------------------------------
# Exclude:
#   Timestamp -> temporal field
#   Label     -> target
#
# Also exclude Tuesday-only identity columns if present.
# ----------------------------------------------------------------------

EXCLUDED_COLUMNS = {
    "Timestamp",
    "Label",
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP"
}

MODEL_FEATURE_COLUMNS = [
    column
    for column in all_columns
    if column not in EXCLUDED_COLUMNS
]

print(
    f"Model input features: "
    f"{len(MODEL_FEATURE_COLUMNS)}"
)

print()

# ----------------------------------------------------------------------
# SAMPLE DATA FROM EACH FILE
# ----------------------------------------------------------------------

sample_frames = []

for index, file_path in enumerate(
    available_files,
    start=1
):

    filename = os.path.basename(file_path)

    print(
        f"[{index}/10] Sampling: {filename}"
    )

    con = duckdb.connect(
        database=":memory:"
    )

    # --------------------------------------------------------------
    # Read only the required model features.
    # --------------------------------------------------------------

    column_sql = ", ".join(
        '"' + col.replace('"', '""') + '"'
        for col in MODEL_FEATURE_COLUMNS
    )

    query = f"""
    SELECT {column_sql}
    FROM read_csv_auto(
        '{file_path.replace("'", "''")}',
        header = TRUE,
        strict_mode = FALSE
    )
    USING SAMPLE {SAMPLE_PER_FILE} ROWS
    """

    sample = con.execute(
        query
    ).fetchdf()

    con.close()

    # --------------------------------------------------------------
    # Convert to numerical values
    # --------------------------------------------------------------

    for column in MODEL_FEATURE_COLUMNS:

        sample[column] = pd.to_numeric(
            sample[column],
            errors="coerce"
        )

    sample_frames.append(sample)

    print(
        f"    Sampled: {len(sample):,} rows ✅"
    )

# ----------------------------------------------------------------------
# COMBINE SAMPLES
# ----------------------------------------------------------------------

sample_df = pd.concat(
    sample_frames,
    ignore_index=True
)

print()
print("=" * 70)
print("SAMPLE CREATED")
print("=" * 70)

print(
    f"Total sampled rows : {len(sample_df):,}"
)

print(
    f"Features           : {len(sample_df.columns)}"
)

# ----------------------------------------------------------------------
# REMOVE COMPLETELY EMPTY FEATURES
# ----------------------------------------------------------------------

usable_columns = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if sample_df[column].notna().sum() > 0
]

sample_df = sample_df[
    usable_columns
]

print(
    f"Usable features    : {len(usable_columns)}"
)

# ----------------------------------------------------------------------
# CALCULATE CORRELATION MATRIX
# ----------------------------------------------------------------------

print()
print("Calculating correlation matrix...")

correlation_matrix = sample_df.corr(
    method="pearson"
)

print("Correlation matrix calculated ✅")

# ----------------------------------------------------------------------
# FIND HIGHLY CORRELATED PAIRS
# ----------------------------------------------------------------------

correlation_pairs = []

columns = correlation_matrix.columns.tolist()

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        feature_a = columns[i]
        feature_b = columns[j]

        correlation = correlation_matrix.loc[
            feature_a,
            feature_b
        ]

        if pd.notna(correlation):

            absolute_correlation = abs(
                correlation
            )

            if (
                absolute_correlation
                >= HIGH_CORRELATION_THRESHOLD
            ):

                correlation_pairs.append({
                    "Feature A": feature_a,
                    "Feature B": feature_b,
                    "Correlation": round(
                        correlation,
                        4
                    ),
                    "Absolute Correlation": round(
                        absolute_correlation,
                        4
                    )
                })

# ----------------------------------------------------------------------
# CREATE RESULT TABLE
# ----------------------------------------------------------------------

redundancy_df = pd.DataFrame(
    correlation_pairs
)

if len(redundancy_df) > 0:

    redundancy_df = redundancy_df.sort_values(
        "Absolute Correlation",
        ascending=False
    ).reset_index(
        drop=True
    )

# ----------------------------------------------------------------------
# DISPLAY RESULTS
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("HIGHLY CORRELATED FEATURE PAIRS")
print("=" * 70)

print(
    f"Threshold: |r| >= "
    f"{HIGH_CORRELATION_THRESHOLD}"
)

print(
    f"Pairs found: "
    f"{len(redundancy_df)}"
)

display(
    redundancy_df
)

# ----------------------------------------------------------------------
# SAVE ANALYSIS RESULT LOCALLY
# ----------------------------------------------------------------------

redundancy_output = (
    "/content/phase3_feature_redundancy.csv"
)

redundancy_df.to_csv(
    redundancy_output,
    index=False
)

# ----------------------------------------------------------------------
# FINAL SUMMARY
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 3.5 SUMMARY")
print("=" * 70)

print(
    f"Files analysed          : 10"
)

print(
    f"Rows sampled            : "
    f"{len(sample_df):,}"
)

print(
    f"Model features analysed : "
    f"{len(usable_columns)}"
)

print(
    f"Highly correlated pairs : "
    f"{len(redundancy_df)}"
)

print()
print(
    "No features were deleted."
)

print(
    "Redundancy analysis saved locally ✅"
)

print()
print(
    "STEP 3.5 COMPLETE ✅"
)

PHASE 3 — STEP 3.5
FEATURE REDUNDANCY ANALYSIS

Phase 3 files available: 10/10
All Phase 3 files found ✅

Columns discovered from Phase 3 schema: 80
Model input features: 78

[1/10] Sampling: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Sampled: 10,000 rows ✅
[2/10] Sampling: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Sampled: 10,000 rows ✅
[3/10] Sampling: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    Sampled: 10,000 rows ✅

SAMPLE CREATED
Total sampled rows : 100,000
Features           : 78
Usable features    : 78

Calculating correlation matrix...
Correlation matrix calculated ✅

HIGHLY CORRELATED FEATURE PAIRS
Threshold: |r| >= 0.95
Pairs found: 46


,Feature A,Feature B,Correlation,Absolute Correlation
0,Tot Fwd Pkts,Subflow Fwd Pkts,1.0000,1.0000
1,Tot Bwd Pkts,Subflow Bwd Pkts,1.0000,1.0000
2,TotLen Fwd Pkts,Subflow Fwd Byts,1.0000,1.0000
3,Bwd Pkt Len Mean,Bwd Seg Size Avg,1.0000,1.0000
4,Fwd Pkt Len Mean,Fwd Seg Size Avg,1.0000,1.0000
5,TotLen Bwd Pkts,Subflow Bwd Byts,1.0000,1.0000
6,Fwd PSH Flags,SYN Flag Cnt,1.0000,1.0000
7,RST Flag Cnt,ECE Flag Cnt,1.0000,1.0000
8,Fwd URG Flags,CWE Flag Count,1.0000,1.0000
9,Bwd Header Len,Subflow Bwd Pkts,0.9993,0.9993



STEP 3.5 SUMMARY
Files analysed          : 10
Rows sampled            : 100,000
Model features analysed : 78
Highly correlated pairs : 46

No features were deleted.
Redundancy analysis saved locally ✅

STEP 3.5 COMPLETE ✅


## 3.6 Feature Selection — Correlation-Based Candidate Analysis

This step identifies features that may be removed because they provide
highly redundant information with other network-traffic features.

The analysis uses the highly correlated feature pairs identified in
Step 3.5, where the absolute Pearson correlation is at least 0.95.

The process considers:

- How frequently each feature appears in highly correlated pairs.
- Which features are strongly redundant with other features.
- Which network-security and forecasting-relevant features should be
  protected from automatic removal.

Features identified here are only **removal candidates**.

No features are deleted at this stage.

The final feature-selection decision will be made after reviewing the
candidate features and their relevance to network attack forecasting.

In [ ]:
# ======================================================================
# PHASE 3 — STEP 3.6
# FEATURE SELECTION — CORRELATION-BASED CANDIDATE ANALYSIS
# ======================================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("PHASE 3 — STEP 3.6")
print("FEATURE SELECTION — CANDIDATE ANALYSIS")
print("=" * 70)
print()

# ----------------------------------------------------------------------
# Load the redundancy results created in Step 3.5
# ----------------------------------------------------------------------

REDUNDANCY_FILE = "/content/phase3_feature_redundancy.csv"

redundancy_df = pd.read_csv(
    REDUNDANCY_FILE
)

print(
    f"Highly correlated pairs loaded: "
    f"{len(redundancy_df)}"
)

# ----------------------------------------------------------------------
# Features already identified as important for network interpretation
# and forecasting.
#
# We will protect these from automatic removal.
# ----------------------------------------------------------------------

PROTECTED_FEATURES = {
    "Flow Duration",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "TotLen Fwd Pkts",
    "TotLen Bwd Pkts",
    "Fwd Pkt Len Mean",
    "Bwd Pkt Len Mean",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Fwd IAT Mean",
    "Bwd IAT Mean",
    "SYN Flag Cnt",
    "RST Flag Cnt",
    "Fwd Header Len",
    "Bwd Header Len"
}

# ----------------------------------------------------------------------
# Count how frequently each feature appears in redundant pairs
# ----------------------------------------------------------------------

feature_frequency = {}

for _, row in redundancy_df.iterrows():

    feature_a = row["Feature A"]
    feature_b = row["Feature B"]

    feature_frequency[feature_a] = (
        feature_frequency.get(feature_a, 0) + 1
    )

    feature_frequency[feature_b] = (
        feature_frequency.get(feature_b, 0) + 1
    )

frequency_df = pd.DataFrame(
    [
        {
            "Feature": feature,
            "Redundant Pair Count": count,
            "Protected":
                feature in PROTECTED_FEATURES
        }
        for feature, count in feature_frequency.items()
    ]
)

frequency_df = frequency_df.sort_values(
    "Redundant Pair Count",
    ascending=False
).reset_index(
    drop=True
)

# ----------------------------------------------------------------------
# Display feature redundancy frequency
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("FEATURE REDUNDANCY FREQUENCY")
print("=" * 70)

display(
    frequency_df
)

# ----------------------------------------------------------------------
# Identify features that can be considered removal candidates
#
# A feature is only a candidate if:
#   1. It participates in a high-correlation pair.
#   2. It is not protected.
#
# These are ONLY candidates — they are not deleted.
# ----------------------------------------------------------------------

candidate_features = [
    feature
    for feature in feature_frequency
    if feature not in PROTECTED_FEATURES
]

candidate_df = frequency_df[
    frequency_df["Feature"].isin(
        candidate_features
    )
].copy()

candidate_df = candidate_df.sort_values(
    "Redundant Pair Count",
    ascending=False
).reset_index(
    drop=True
)

# ----------------------------------------------------------------------
# Display candidate features
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("FEATURE REMOVAL CANDIDATES")
print("=" * 70)

print(
    f"Candidate features: "
    f"{len(candidate_df)}"
)

display(
    candidate_df
)

# ----------------------------------------------------------------------
# Save candidate list
# ----------------------------------------------------------------------

candidate_output = (
    "/content/phase3_feature_selection_candidates.csv"
)

candidate_df.to_csv(
    candidate_output,
    index=False
)

# ----------------------------------------------------------------------
# Summary
# ----------------------------------------------------------------------

print()
print("=" * 70)
print("STEP 3.6 CANDIDATE ANALYSIS COMPLETE")
print("=" * 70)

print(
    f"Highly correlated pairs : "
    f"{len(redundancy_df)}"
)

print(
    f"Features appearing in "
    f"redundant pairs          : "
    f"{len(feature_frequency)}"
)

print(
    f"Protected features        : "
    f"{len(PROTECTED_FEATURES)}"
)

print(
    f"Removal candidates        : "
    f"{len(candidate_df)}"
)

print()
print(
    "IMPORTANT: No features have been deleted."
)

print(
    "Candidate list saved locally ✅"
)

print()
print("NEXT: FINAL FEATURE-SELECTION DECISION")
print("=" * 70)

PHASE 3 — STEP 3.6
FEATURE SELECTION — CANDIDATE ANALYSIS

Highly correlated pairs loaded: 46

FEATURE REDUNDANCY FREQUENCY


,Feature,Redundant Pair Count,Protected
0,Tot Fwd Pkts,5,True
1,Subflow Fwd Pkts,5,False
2,Subflow Fwd Byts,5,False
3,TotLen Fwd Pkts,5,True
4,Fwd Header Len,5,True
5,Fwd Act Data Pkts,5,False
6,TotLen Bwd Pkts,4,True
7,Bwd Header Len,4,True
8,Subflow Bwd Pkts,4,False
9,Tot Bwd Pkts,4,True



FEATURE REMOVAL CANDIDATES
Candidate features: 27


,Feature,Redundant Pair Count,Protected
0,Subflow Fwd Pkts,5,False
1,Subflow Fwd Byts,5,False
2,Fwd Act Data Pkts,5,False
3,Subflow Bwd Pkts,4,False
4,Subflow Bwd Byts,4,False
5,Flow IAT Min,3,False
6,Fwd IAT Min,3,False
7,Idle Mean,2,False
8,Idle Max,2,False
9,Idle Min,2,False



STEP 3.6 CANDIDATE ANALYSIS COMPLETE
Highly correlated pairs : 46
Features appearing in redundant pairs          : 42
Protected features        : 17
Removal candidates        : 27

IMPORTANT: No features have been deleted.
Candidate list saved locally ✅

NEXT: FINAL FEATURE-SELECTION DECISION


## 3.6.1 Final Feature Set Definition

Based on the feature redundancy analysis, highly redundant features are
reviewed before removal.

The objective is to retain representative network-traffic features while
avoiding unnecessary duplication.

Features are removed only when they are clearly redundant or represent
an alternative representation of information already retained.

The final feature list will be used for the Phase 3 dataset.

This step defines the feature set only; the Phase 3 source datasets are
not modified.

In [ ]:
# ================================================================
# PHASE 3 — STEP 3.6.1
# FINAL FEATURE SET DEFINITION
# ================================================================

print("=" * 70)
print("PHASE 3 — STEP 3.6.1")
print("FINAL FEATURE SET DEFINITION")
print("=" * 70)

import os
import glob
import pandas as pd

# ------------------------------------------------
# 1. Locate a Phase 3 dataset file
# ------------------------------------------------

phase3_files = sorted(
    glob.glob("/content/phase3_working/*.csv")
)

if not phase3_files:
    raise FileNotFoundError(
        "No Phase 3 files found in /content/phase3_working/"
    )

sample_file = phase3_files[0]

print(f"\nSample Phase 3 file:")
print(os.path.basename(sample_file))

# ------------------------------------------------
# 2. Read only the header
# ------------------------------------------------

sample_columns = pd.read_csv(
    sample_file,
    nrows=0
).columns.tolist()

print(f"\nTotal columns in sample file: {len(sample_columns)}")

# ------------------------------------------------
# 3. Define non-model columns
# ------------------------------------------------

NON_MODEL_COLUMNS = {
    "Timestamp",
    "Label",
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP"
}

# ------------------------------------------------
# 4. Automatically derive the 78 common model
#    features from the Phase 3 schema
# ------------------------------------------------

ALL_MODEL_FEATURES = [
    col for col in sample_columns
    if col not in NON_MODEL_COLUMNS
]

print(f"Detected model features: {len(ALL_MODEL_FEATURES)}")

# ------------------------------------------------
# 5. Validate model feature count
# ------------------------------------------------

if len(ALL_MODEL_FEATURES) != 78:
    print("\nDetected features:")
    for i, feature in enumerate(ALL_MODEL_FEATURES, 1):
        print(f"{i:2d}. {feature}")

    raise RuntimeError(
        f"Expected 78 model features, "
        f"found {len(ALL_MODEL_FEATURES)}. "
        f"Check the Phase 3 input schema."
    )

# ------------------------------------------------
# 6. Features selected for removal
#    Based on Phase 3.5 redundancy analysis
# ------------------------------------------------

FEATURES_TO_REMOVE = [
    "Subflow Fwd Pkts",
    "Subflow Fwd Byts",
    "Fwd Act Data Pkts",
    "Subflow Bwd Pkts",
    "Subflow Bwd Byts",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Fwd Seg Size Avg",
    "Bwd Seg Size Avg",
    "Fwd Pkts/s"
]

# ------------------------------------------------
# 7. Validate removal list
# ------------------------------------------------

missing_removals = [
    feature
    for feature in FEATURES_TO_REMOVE
    if feature not in ALL_MODEL_FEATURES
]

if missing_removals:
    raise RuntimeError(
        "The following removal candidates are not present "
        f"in the detected 78-feature schema:\n{missing_removals}"
    )

# ------------------------------------------------
# 8. Create final 68-feature set
# ------------------------------------------------

FINAL_MODEL_FEATURES = [
    feature
    for feature in ALL_MODEL_FEATURES
    if feature not in FEATURES_TO_REMOVE
]

# ------------------------------------------------
# 9. Final validation
# ------------------------------------------------

if len(FEATURES_TO_REMOVE) != 10:
    raise RuntimeError(
        f"Expected 10 removals, "
        f"found {len(FEATURES_TO_REMOVE)}"
    )

if len(FINAL_MODEL_FEATURES) != 68:
    raise RuntimeError(
        f"Expected 68 final model features, "
        f"found {len(FINAL_MODEL_FEATURES)}"
    )

# ------------------------------------------------
# 10. Save feature lists
# ------------------------------------------------

pd.DataFrame({
    "Feature": ALL_MODEL_FEATURES
}).to_csv(
    "/content/phase3_all_78_features.csv",
    index=False
)

pd.DataFrame({
    "Removed_Feature": FEATURES_TO_REMOVE
}).to_csv(
    "/content/phase3_removed_10_features.csv",
    index=False
)

pd.DataFrame({
    "Final_Feature": FINAL_MODEL_FEATURES
}).to_csv(
    "/content/phase3_final_68_features.csv",
    index=False
)

# ------------------------------------------------
# 11. Display final summary
# ------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE SELECTION SUMMARY")
print("=" * 70)

print(f"Original model features : {len(ALL_MODEL_FEATURES)}")
print(f"Features removed        : {len(FEATURES_TO_REMOVE)}")
print(f"Final model features    : {len(FINAL_MODEL_FEATURES)}")

print("\nRemoved features:")
for feature in FEATURES_TO_REMOVE:
    print(f"  - {feature}")

print("\nFinal 68 features:")
for i, feature in enumerate(FINAL_MODEL_FEATURES, 1):
    print(f"{i:2d}. {feature}")

print("\n" + "=" * 70)
print("STEP 3.6.1 COMPLETED")
print("No dataset files were modified.")
print("=" * 70)

PHASE 3 — STEP 3.6.1
FINAL FEATURE SET DEFINITION

Sample Phase 3 file:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv

Total columns in sample file: 80
Detected model features: 78

FEATURE SELECTION SUMMARY
Original model features : 78
Features removed        : 10
Final model features    : 68

Removed features:
  - Subflow Fwd Pkts
  - Subflow Fwd Byts
  - Fwd Act Data Pkts
  - Subflow Bwd Pkts
  - Subflow Bwd Byts
  - Flow IAT Min
  - Fwd IAT Min
  - Fwd Seg Size Avg
  - Bwd Seg Size Avg
  - Fwd Pkts/s

Final 68 features:
 1. Dst Port
 2. Protocol
 3. Flow Duration
 4. Tot Fwd Pkts
 5. Tot Bwd Pkts
 6. TotLen Fwd Pkts
 7. TotLen Bwd Pkts
 8. Fwd Pkt Len Max
 9. Fwd Pkt Len Min
10. Fwd Pkt Len Mean
11. Fwd Pkt Len Std
12. Bwd Pkt Len Max
13. Bwd Pkt Len Min
14. Bwd Pkt Len Mean
15. Bwd Pkt Len Std
16. Flow Byts/s
17. Flow Pkts/s
18. Flow IAT Mean
19. Flow IAT Std
20. Flow IAT Max
21. Fwd IAT Tot
22. Fwd IAT Mean
23. Fwd IAT Std
24. Fwd IAT Max
25. Bwd IAT Tot
26. Bwd IAT Mean
27. Bwd

In [ ]:
# ================================================================
# PHASE 3 — STEP 3.6.2
# APPLY FINAL FEATURE SELECTION
# CREATE FINAL PHASE 3 DATASET
# ================================================================

import os
import glob
import duckdb
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.6.2")
print("APPLY FINAL FEATURE SELECTION")
print("=" * 70)

# ------------------------------------------------
# 1. Configuration
# ------------------------------------------------

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"
HF_INPUT_DIR = "processed/phase3"
HF_OUTPUT_DIR = "processed/phase3/final"

LOCAL_INPUT_DIR = "/content/phase3_working"
LOCAL_OUTPUT_DIR = "/content/phase3_final"

os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. Load the validated feature lists from Step 3.6.1
# ------------------------------------------------

all_features_path = "/content/phase3_all_78_features.csv"
removed_features_path = "/content/phase3_removed_10_features.csv"

if not os.path.exists(all_features_path):
    raise FileNotFoundError(
        "phase3_all_78_features.csv not found. "
        "Run Step 3.6.1 first."
    )

if not os.path.exists(removed_features_path):
    raise FileNotFoundError(
        "phase3_removed_10_features.csv not found. "
        "Run Step 3.6.1 first."
    )

ALL_MODEL_FEATURES = pd.read_csv(
    all_features_path
)["Feature"].tolist()

FEATURES_TO_REMOVE = pd.read_csv(
    removed_features_path
)["Removed_Feature"].tolist()

FINAL_MODEL_FEATURES = [
    feature
    for feature in ALL_MODEL_FEATURES
    if feature not in FEATURES_TO_REMOVE
]

# ------------------------------------------------
# 3. Validate feature configuration
# ------------------------------------------------

if len(ALL_MODEL_FEATURES) != 78:
    raise RuntimeError(
        f"Expected 78 original features, found {len(ALL_MODEL_FEATURES)}"
    )

if len(FEATURES_TO_REMOVE) != 10:
    raise RuntimeError(
        f"Expected 10 removed features, found {len(FEATURES_TO_REMOVE)}"
    )

if len(FINAL_MODEL_FEATURES) != 68:
    raise RuntimeError(
        f"Expected 68 final features, found {len(FINAL_MODEL_FEATURES)}"
    )

# Timestamp and Label are retained for future temporal modelling.
RETAIN_COLUMNS = ["Timestamp", "Label"]

FINAL_COLUMNS = RETAIN_COLUMNS + FINAL_MODEL_FEATURES

print(f"\nOriginal model features : {len(ALL_MODEL_FEATURES)}")
print(f"Removed features        : {len(FEATURES_TO_REMOVE)}")
print(f"Final model features    : {len(FINAL_MODEL_FEATURES)}")
print(f"Final dataset columns   : {len(FINAL_COLUMNS)}")

# ------------------------------------------------
# 4. Find local Phase 3 files
# ------------------------------------------------

local_files = sorted(
    glob.glob(os.path.join(LOCAL_INPUT_DIR, "*.csv"))
)

print(f"\nLocal Phase 3 files found: {len(local_files)}")

# ------------------------------------------------
# 5. If local files are unavailable, download them
#    from the Phase 3 HF directory.
# ------------------------------------------------

if len(local_files) != 10:

    print("\nLocal files incomplete.")
    print("Downloading Phase 3 files from Hugging Face...")

    api = HfApi()

    repo_files = api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="dataset"
    )

    hf_phase3_files = sorted([
        f for f in repo_files
        if f.startswith(HF_INPUT_DIR + "/")
        and f.endswith(".csv")
        and "/final/" not in f
    ])

    if len(hf_phase3_files) != 10:
        raise RuntimeError(
            f"Expected 10 Phase 3 files on Hugging Face, "
            f"found {len(hf_phase3_files)}"
        )

    local_files = []

    for hf_file in hf_phase3_files:

        downloaded = hf_hub_download(
            repo_id=HF_REPO,
            filename=hf_file,
            repo_type="dataset",
            local_dir="/content/hf_phase3_download"
        )

        local_files.append(downloaded)

    local_files = sorted(local_files)

print(f"Files ready for processing: {len(local_files)}")

if len(local_files) != 10:
    raise RuntimeError(
        "Exactly 10 Phase 3 files are required."
    )

# ------------------------------------------------
# 6. Process each file using DuckDB
# ------------------------------------------------

con = duckdb.connect()

results = []

for index, input_file in enumerate(local_files, 1):

    filename = os.path.basename(input_file)
    output_file = os.path.join(
        LOCAL_OUTPUT_DIR,
        filename
    )

    print("\n" + "-" * 70)
    print(f"[{index}/10] Processing:")
    print(filename)

    # ------------------------------------------------
    # Read only the header first
    # ------------------------------------------------

    schema_df = con.execute(
        f"""
        DESCRIBE SELECT *
        FROM read_csv_auto(
            '{input_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        """
    ).df()

    available_columns = schema_df["column_name"].tolist()

    # Check required columns
    missing_columns = [
        col for col in FINAL_COLUMNS
        if col not in available_columns
    ]

    if missing_columns:
        raise RuntimeError(
            f"Missing required columns in {filename}: "
            f"{missing_columns}"
        )

    # ------------------------------------------------
    # Build safe SQL column selection
    # ------------------------------------------------

    quoted_columns = ", ".join(
        '"' + col.replace('"', '""') + '"'
        for col in FINAL_COLUMNS
    )

    # ------------------------------------------------
    # Create final dataset
    #
    # No rows are removed here.
    # Only columns are selected.
    # ------------------------------------------------

    query = f"""
        COPY (
            SELECT
                {quoted_columns}
            FROM read_csv_auto(
                '{input_file}',
                header=true,
                all_varchar=true,
                strict_mode=false,
                ignore_errors=true
            )
        )
        TO '{output_file}'
        (HEADER, DELIMITER ',');
    """

    con.execute(query)

    # ------------------------------------------------
    # Validate output
    # ------------------------------------------------

    output_schema = con.execute(
        f"""
        DESCRIBE SELECT *
        FROM read_csv_auto(
            '{output_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        """
    ).df()

    output_columns = output_schema["column_name"].tolist()

    row_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{output_file}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        """
    ).fetchone()[0]

    if output_columns != FINAL_COLUMNS:
        raise RuntimeError(
            f"Column validation failed for {filename}"
        )

    print(f"Rows preserved : {row_count:,}")
    print(f"Columns        : {len(output_columns)}")
    print("Validation     : PASS")

    results.append({
        "File": filename,
        "Rows": row_count,
        "Columns": len(output_columns),
        "Status": "PASS"
    })

con.close()

# ------------------------------------------------
# 7. Overall validation
# ------------------------------------------------

results_df = pd.DataFrame(results)

expected_total_rows = 15_822_221

actual_total_rows = int(
    results_df["Rows"].sum()
)

print("\n" + "=" * 70)
print("PHASE 3 FINAL DATASET SUMMARY")
print("=" * 70)

print(f"Files processed       : {len(results_df)}/10")
print(f"Total rows            : {actual_total_rows:,}")
print(f"Expected rows         : {expected_total_rows:,}")
print(f"Final columns         : {len(FINAL_COLUMNS)}")
print(f"Model features        : {len(FINAL_MODEL_FEATURES)}")
print(f"Timestamp retained    : {'Timestamp' in FINAL_COLUMNS}")
print(f"Label retained        : {'Label' in FINAL_COLUMNS}")

if actual_total_rows != expected_total_rows:
    raise RuntimeError(
        f"Row-count mismatch! "
        f"Expected {expected_total_rows:,}, "
        f"found {actual_total_rows:,}"
    )

if not all(results_df["Columns"] == 70):
    raise RuntimeError(
        "Not all final Phase 3 files contain exactly 70 columns."
    )

print("\nPer-file results:")
print(results_df.to_string(index=False))

# ------------------------------------------------
# 8. Save local validation report
# ------------------------------------------------

results_df.to_csv(
    "/content/phase3_final_validation_summary.csv",
    index=False
)

# ------------------------------------------------
# 9. Upload final datasets to Hugging Face
# ------------------------------------------------

print("\n" + "=" * 70)
print("UPLOADING FINAL PHASE 3 DATASETS")
print("=" * 70)

api = HfApi()

for index, output_file in enumerate(
    sorted(glob.glob(os.path.join(LOCAL_OUTPUT_DIR, "*.csv"))),
    1
):

    filename = os.path.basename(output_file)

    print(
        f"\n[{index}/10] Uploading {filename}..."
    )

    api.upload_file(
        path_or_fileobj=output_file,
        path_in_repo=f"{HF_OUTPUT_DIR}/{filename}",
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=(
            "Phase 3 final feature selection dataset"
        )
    )

    print("Upload: PASS")

# ------------------------------------------------
# 10. Final completion message
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 3.6.2 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFinal Phase 3 dataset:")
print("  10 files")
print(f"  {actual_total_rows:,} rows")
print("  70 columns")
print("  68 model features")
print("  Timestamp")
print("  Label")

print("\nHF location:")
print(f"  {HF_OUTPUT_DIR}/")

print("\nPhase 2 datasets remain unchanged.")
print("Phase 3 intermediate datasets remain unchanged.")
print("Final feature-selected datasets created separately.")

PHASE 3 — STEP 3.6.2
APPLY FINAL FEATURE SELECTION

Original model features : 78
Removed features        : 10
Final model features    : 68
Final dataset columns   : 70

Local Phase 3 files found: 10
Files ready for processing: 10

----------------------------------------------------------------------
[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows preserved : 1,043,116
Columns        : 70
Validation     : PASS

----------------------------------------------------------------------
[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows preserved : 900,988
Columns        : 70
Validation     : PASS

----------------------------------------------------------------------
[3/10] Processing:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


... (58 outputs left out)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ficForML_CICFlowMeter.csv:   1%|1         | 2.76MB /  198MB            

Upload: PASS

STEP 3.6.2 COMPLETED SUCCESSFULLY

Final Phase 3 dataset:
  10 files
  15,822,221 rows
  70 columns
  68 model features
  Timestamp
  Label

HF location:
  processed/phase3/final/

Phase 2 datasets remain unchanged.
Phase 3 intermediate datasets remain unchanged.
Final feature-selected datasets created separately.


In [4]:
# ================================================================
# PHASE 3 — STEP 3.7.1
# HUGGING FACE AUTHENTICATION CHECK
# ================================================================

from huggingface_hub import login, whoami, HfApi

print("=" * 70)
print("PHASE 3 — STEP 3.7.1")
print("HUGGING FACE AUTHENTICATION")
print("=" * 70)

# Open the secure Hugging Face login widget.
# Do NOT paste your token into the notebook code or into chat.
login()

# Verify the authenticated account
user_info = whoami()

username = user_info.get("name")

print("\nAuthenticated Hugging Face account:")
print(f"  {username}")

if username != "PrakulShetty":
    raise RuntimeError(
        f"Wrong Hugging Face account detected: {username}"
    )

# ------------------------------------------------
# Verify repository access
# ------------------------------------------------

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"

api = HfApi()

repo_info = api.dataset_info(
    repo_id=HF_REPO
)

print("\nDataset repository:")
print(f"  {repo_info.id}")

if repo_info.id.lower() != HF_REPO.lower():
    raise RuntimeError(
        "Unexpected Hugging Face repository."
    )

print("\n" + "=" * 70)
print("HUGGING FACE AUTHENTICATION: PASS")
print("DATASET REPOSITORY ACCESS: PASS")
print("=" * 70)

PHASE 3 — STEP 3.7.1
HUGGING FACE AUTHENTICATION

Authenticated Hugging Face account:
  PrakulShetty

Dataset repository:
  PrakulShetty/NetVision-AI-SIH26153

HUGGING FACE AUTHENTICATION: PASS
DATASET REPOSITORY ACCESS: PASS


In [5]:
# ================================================================
# PHASE 3 — STEP 3.7
# FINAL HUGGING FACE DATASET VALIDATION
# ================================================================

import os
import duckdb
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

print("=" * 70)
print("PHASE 3 — STEP 3.7")
print("FINAL HUGGING FACE DATASET VALIDATION")
print("=" * 70)

# ------------------------------------------------
# 1. Configuration
# ------------------------------------------------

HF_REPO = "PrakulShetty/NetVision-AI-SIH26153"
HF_FINAL_DIR = "processed/phase3/final"

EXPECTED_FILES = 10
EXPECTED_TOTAL_ROWS = 15_822_221
EXPECTED_TOTAL_COLUMNS = 70
EXPECTED_MODEL_FEATURES = 68

# ------------------------------------------------
# 2. Connect to Hugging Face
# ------------------------------------------------

api = HfApi()

print("\nConnecting to Hugging Face...")

repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset"
)

print("Repository access: PASS")

# ------------------------------------------------
# 3. Find final Phase 3 datasets
# ------------------------------------------------

hf_final_files = sorted([
    file
    for file in repo_files
    if file.startswith(HF_FINAL_DIR + "/")
    and file.endswith(".csv")
])

print(
    f"\nFinal Phase 3 files found: "
    f"{len(hf_final_files)}"
)

if len(hf_final_files) != EXPECTED_FILES:
    raise RuntimeError(
        f"Expected {EXPECTED_FILES} final Phase 3 files, "
        f"found {len(hf_final_files)}"
    )

print("\nFinal files on Hugging Face:")

for file in hf_final_files:
    print(f"  ✓ {file}")

# ------------------------------------------------
# 4. Download files for validation
# ------------------------------------------------

DOWNLOAD_DIR = "/content/phase3_hf_validation"

os.makedirs(
    DOWNLOAD_DIR,
    exist_ok=True
)

local_files = []

print("\nDownloading final datasets for validation...")

for index, hf_file in enumerate(
    hf_final_files,
    1
):

    filename = os.path.basename(hf_file)

    print(
        f"[{index}/{EXPECTED_FILES}] "
        f"{filename}"
    )

    local_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=hf_file,
        repo_type="dataset",
        local_dir=DOWNLOAD_DIR
    )

    local_files.append(local_path)

print("\nDownload status: PASS")

# ------------------------------------------------
# 5. Create DuckDB connection
# ------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------
# 6. Determine expected schema from first final file
# ------------------------------------------------

reference_file = sorted(local_files)[0]

print("\n" + "-" * 70)
print("READING REFERENCE FINAL DATASET")
print("-" * 70)

reference_schema = con.execute(
    f"""
    DESCRIBE SELECT *
    FROM read_csv_auto(
        '{reference_file}',
        header=true,
        all_varchar=true,
        strict_mode=false,
        ignore_errors=true
    )
    """
).df()

EXPECTED_COLUMN_LIST = (
    reference_schema["column_name"].tolist()
)

reference_column_count = len(
    EXPECTED_COLUMN_LIST
)

print(
    f"Reference file: "
    f"{os.path.basename(reference_file)}"
)

print(
    f"Reference columns: "
    f"{reference_column_count}"
)

# ------------------------------------------------
# 7. Validate reference schema
# ------------------------------------------------

if reference_column_count != EXPECTED_TOTAL_COLUMNS:
    raise RuntimeError(
        f"Reference dataset has "
        f"{reference_column_count} columns. "
        f"Expected {EXPECTED_TOTAL_COLUMNS}."
    )

if "Timestamp" not in EXPECTED_COLUMN_LIST:
    raise RuntimeError(
        "Timestamp column missing."
    )

if "Label" not in EXPECTED_COLUMN_LIST:
    raise RuntimeError(
        "Label column missing."
    )

reference_model_features = [
    col
    for col in EXPECTED_COLUMN_LIST
    if col not in ["Timestamp", "Label"]
]

if len(reference_model_features) != EXPECTED_MODEL_FEATURES:
    raise RuntimeError(
        f"Expected {EXPECTED_MODEL_FEATURES} model features, "
        f"found {len(reference_model_features)}"
    )

print("Reference schema: PASS")
print(f"Model features: {len(reference_model_features)}")
print("Timestamp: PRESENT")
print("Label: PRESENT")

# ------------------------------------------------
# 8. Expected row counts from Phase 3
# ------------------------------------------------

EXPECTED_ROW_COUNTS = {
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv": 1_043_116,
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv": 900_988,
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_960,
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv": 7_948_746,
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv": 331_027,
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv": 1_046_154,
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv": 1_045_288,
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": 822_942,
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv": 1_031_018,
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv": 606_982
}

# ------------------------------------------------
# 9. Validate every final HF dataset
# ------------------------------------------------

validation_results = []

for index, file_path in enumerate(
    sorted(local_files),
    1
):

    filename = os.path.basename(file_path)

    print("\n" + "-" * 70)
    print(
        f"[{index}/{EXPECTED_FILES}] "
        f"VALIDATING:"
    )
    print(filename)

    # ------------------------------------------------
    # Schema
    # ------------------------------------------------

    schema_df = con.execute(
        f"""
        DESCRIBE SELECT *
        FROM read_csv_auto(
            '{file_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        """
    ).df()

    columns = schema_df["column_name"].tolist()

    schema_pass = (
        columns == EXPECTED_COLUMN_LIST
        and len(columns) == EXPECTED_TOTAL_COLUMNS
    )

    if not schema_pass:
        missing = [
            col
            for col in EXPECTED_COLUMN_LIST
            if col not in columns
        ]

        unexpected = [
            col
            for col in columns
            if col not in EXPECTED_COLUMN_LIST
        ]

        print("Schema: FAIL")
        print("Missing:", missing)
        print("Unexpected:", unexpected)

        raise RuntimeError(
            f"Schema validation failed for {filename}"
        )

    # ------------------------------------------------
    # Row count
    # ------------------------------------------------

    row_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        """
    ).fetchone()[0]

    expected_rows = EXPECTED_ROW_COUNTS.get(
        filename
    )

    if expected_rows is None:
        raise RuntimeError(
            f"No expected row count registered for {filename}"
        )

    row_pass = (
        row_count == expected_rows
    )

    # ------------------------------------------------
    # Timestamp validation
    # ------------------------------------------------

    invalid_timestamp_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        WHERE "Timestamp" IS NULL
           OR TRY_CAST("Timestamp" AS TIMESTAMP) IS NULL
        """
    ).fetchone()[0]

    timestamp_pass = (
        invalid_timestamp_count == 0
    )

    # ------------------------------------------------
    # Label validation
    # ------------------------------------------------

    empty_label_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=true,
            all_varchar=true,
            strict_mode=false,
            ignore_errors=true
        )
        WHERE "Label" IS NULL
           OR TRIM("Label") = ''
        """
    ).fetchone()[0]

    label_pass = (
        empty_label_count == 0
    )

    # ------------------------------------------------
    # Final file status
    # ------------------------------------------------

    file_pass = (
        schema_pass
        and row_pass
        and timestamp_pass
        and label_pass
    )

    print(
        f"Rows                  : "
        f"{row_count:,}"
    )

    print(
        f"Expected rows         : "
        f"{expected_rows:,}"
    )

    print(
        f"Columns               : "
        f"{len(columns)}"
    )

    print(
        f"Invalid timestamps    : "
        f"{invalid_timestamp_count:,}"
    )

    print(
        f"Empty labels          : "
        f"{empty_label_count:,}"
    )

    print(
        f"Schema                : "
        f"{'PASS' if schema_pass else 'FAIL'}"
    )

    print(
        f"Row count             : "
        f"{'PASS' if row_pass else 'FAIL'}"
    )

    print(
        f"Timestamp             : "
        f"{'PASS' if timestamp_pass else 'FAIL'}"
    )

    print(
        f"Label                 : "
        f"{'PASS' if label_pass else 'FAIL'}"
    )

    print(
        f"Overall               : "
        f"{'PASS' if file_pass else 'FAIL'}"
    )

    if not file_pass:
        raise RuntimeError(
            f"Final HF validation failed for {filename}"
        )

    validation_results.append({
        "File": filename,
        "Rows": row_count,
        "Expected_Rows": expected_rows,
        "Columns": len(columns),
        "Invalid_Timestamps": invalid_timestamp_count,
        "Empty_Labels": empty_label_count,
        "Status": "PASS"
    })

# ------------------------------------------------
# 10. Close DuckDB
# ------------------------------------------------

con.close()

# ------------------------------------------------
# 11. Overall validation
# ------------------------------------------------

results_df = pd.DataFrame(
    validation_results
)

total_rows = int(
    results_df["Rows"].sum()
)

total_expected_rows = int(
    results_df["Expected_Rows"].sum()
)

print("\n" + "=" * 70)
print("PHASE 3 — FINAL HUGGING FACE VALIDATION SUMMARY")
print("=" * 70)

print(
    f"Files validated       : "
    f"{len(results_df)}/{EXPECTED_FILES}"
)

print(
    f"Total rows            : "
    f"{total_rows:,}"
)

print(
    f"Expected total rows   : "
    f"{total_expected_rows:,}"
)

print(
    f"Columns per file      : "
    f"{EXPECTED_TOTAL_COLUMNS}"
)

print(
    f"Model features        : "
    f"{EXPECTED_MODEL_FEATURES}"
)

print(
    f"Invalid timestamps    : "
    f"{results_df['Invalid_Timestamps'].sum():,}"
)

print(
    f"Empty labels          : "
    f"{results_df['Empty_Labels'].sum():,}"
)

# ------------------------------------------------
# 12. Final assertions
# ------------------------------------------------

if len(results_df) != EXPECTED_FILES:
    raise RuntimeError(
        "Expected 10 validated files."
    )

if total_rows != EXPECTED_TOTAL_ROWS:
    raise RuntimeError(
        f"Total rows mismatch: "
        f"{total_rows:,} != "
        f"{EXPECTED_TOTAL_ROWS:,}"
    )

if total_expected_rows != EXPECTED_TOTAL_ROWS:
    raise RuntimeError(
        "Expected row-count configuration is incorrect."
    )

if not all(
    results_df["Columns"] == EXPECTED_TOTAL_COLUMNS
):
    raise RuntimeError(
        "Column validation failed."
    )

if results_df["Invalid_Timestamps"].sum() != 0:
    raise RuntimeError(
        "Invalid timestamps detected."
    )

if results_df["Empty_Labels"].sum() != 0:
    raise RuntimeError(
        "Empty labels detected."
    )

# ------------------------------------------------
# 13. Save validation report
# ------------------------------------------------

REPORT_PATH = (
    "/content/PHASE_3_FINAL_HF_VALIDATION_REPORT.csv"
)

results_df.to_csv(
    REPORT_PATH,
    index=False
)

# ------------------------------------------------
# 14. Final success
# ------------------------------------------------

print("\nPer-file results:")
print(
    results_df.to_string(index=False)
)

print("\n" + "=" * 70)
print("PHASE 3 — FINAL VALIDATION PASSED")
print("=" * 70)

print("""
✓ 10/10 final datasets verified from Hugging Face
✓ 15,822,221 total rows
✓ 70 columns per dataset
✓ 68 model features
✓ Timestamp retained
✓ Label retained
✓ Row counts match Phase 3
✓ No invalid timestamps
✓ No empty labels
✓ Final datasets are ready for Phase 4

================================================
PHASE 3 IS OFFICIALLY COMPLETE
================================================

NEXT:
PHASE 4 — TEMPORAL REPRESENTATION
""")

print("=" * 70)

PHASE 3 — STEP 3.7
FINAL HUGGING FACE DATASET VALIDATION

Connecting to Hugging Face...
Repository access: PASS

Final Phase 3 files found: 10

Final files on Hugging Face:
  ✓ processed/phase3/final/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  ✓ processed/phase3/final/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


processed/phase3/final/Friday-02-03-2018(…): reconstructing file:   0%|          |  0.00B /  354MB            

processed/phase3/final/Friday-02-03-2018(…): downloading bytes:           |  0.00B            

[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


processed/phase3/final/Friday-16-02-2018(…): reconstructing file:   0%|          |  0.00B /  288MB            

processed/phase3/final/Friday-16-02-2018(…): downloading bytes:           |  0.00B            

... (47 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                  : 1,031,018
Expected rows         : 1,031,018
Columns               : 70
Invalid timestamps    : 0
Empty labels          : 0
Schema                : PASS
Row count             : PASS
Timestamp             : PASS
Label                 : PASS
Overall               : PASS

----------------------------------------------------------------------
[10/10] VALIDATING:
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                  : 606,982
Expected rows         : 606,982
Columns               : 70
Invalid timestamps    : 0
Empty labels          : 0
Schema                : PASS
Row count             : PASS
Timestamp             : PASS
Label                 : PASS
Overall               : PASS

PHASE 3 — FINAL HUGGING FACE VALIDATION SUMMARY
Files validated       : 10/10
Total rows            : 15,822,221
Expected total rows   : 15,822,221
Columns per file      : 70
Model features        : 68
Invalid timestamps    : 0
Empty labels          : 0

Per-file results:
                                              File    Rows  Expected_Rows  Columns  Invalid_Timestamps  Empty_Labels Status
   Friday-02-03-2018_TrafficForML_CICFlowMeter.csv 1043116        1043116       70                   0             0   PASS
   Friday-16-02-2018_TrafficForML_CICFlowMeter.csv  900988         900988       70                   0             0   PASS
   Friday-23-02-2018_TrafficForML_CICFlowMeter.csv 1045960        